# Week 2：ESP32-S3 開發環境與數位輸入輸出實驗

日期：2026-09-16

本章以 ESP32-S3 開發板為實驗平台，完成開發環境確認、程式編譯與上傳、
GPIO 按鈕輸入、數位輸出命令及斷電通斷驗證。

> 實機驗證門檻：第一片實物為`YD-ESP32-S3 Type-A V1.5`，模組是
> `ESP32-S3-WROOM-1 N16R8`；板上絲印可見GPIO4與GPIO5，但
> `docs/hardware_state.md`已將`BOARD-T01`標為`basic-pass`：無外接線的板卡辨識
> 程式已完成Compile、Upload、Serial、Flash與PSRAM基本驗證。GPIO4候選輸入已完成五次按下／放開事件；GPIO5目前只有程式輸出命令，
> 實際LOW／HIGH電壓尚未量測，因此兩者尚未成為學生固定接線答案。
> 無外接線的板卡辨識程式可以繼續Verify、Upload及觀察Serial；涉及外部GPIO的
> 範例程式則把腳位預設為`-1`，在教師公布hardware profile前不會啟用GPIO，
> 也不得自行接線試錯。

> Week 2範圍在三項收尾練習與安全復原後結束。散裝電阻、3V3／5Vin、
> GPIO5實際LOW／HIGH電壓及Reset後LOW電壓證據，移到Week 3感測器接線前完成。


## 一、Unit Overview

### Teaching Objectives

By the end of this unit, students will be able to:

1. Identify the ESP32-S3 USB, GPIO, GND, BOOT, and RESET connections used in the
   Week 2 circuit.
2. Configure Arduino IDE, compile and upload a program, and interpret Serial Monitor
   output.
3. Build and test a push-button input with `INPUT_PULLUP`, and distinguish an input
   reading from an output command.
4. Use a multimeter in an unpowered circuit to identify connected breadboard nodes and
   the open and closed states of a tactile pushbutton.
5. Explain switch bounce, trace a software debounce algorithm, and use controlled evidence
   to compare debounce intervals.

### Teaching Content

This unit introduces the development and evidence workflow for a small embedded system
with an ESP32-S3. Students identify the exact board, configure the development environment,
compile and upload firmware, and use Serial output to determine what the running program
has actually reported.

The hardware activity uses a breadboard and a tactile pushbutton to study nodes, GND,
`INPUT_PULLUP`, HIGH, LOW, open circuits, and closed circuits. Students verify the
unpowered button path with a multimeter, distinguish raw input changes from accepted stable
states, trace a non-blocking debounce algorithm, and compare debounce intervals without
changing the circuit. Voltage and loose-resistor measurements are intentionally continued
at the beginning of Week 3, before sensors are connected.

### 核心實驗流程

| 階段 | 開始狀態 | 實驗內容 | 完成條件 |
|---:|---|---|---|
| 1 | 板子未接 USB | 環境與實物辨識 | 找到 USB、BOOT、RESET、候選GPIO4、候選GPIO5、GND |
| 2 | 只接 USB | 設定 Board 與 Port | IDE 顯示正確 Board、Port 與 N16R8 設定 |
| 3 | 只接 USB | Upload 與 Serial | log 顯示本組組別及 `version=2` |
| 4 | USB 已拔除 | 麵包板、按鈕、TPO、TPG 接線 | 依profile與接線表逐線確認；正向、反向各檢查一次 |
| 5 | 已公布profile且接線已確認 | profile輸入與測試輸出命令 | 五次按下／放開事件完全對應 |
| 6 | USB 已拔除 | 麵包板與按鈕通斷 | 同列、跨槽、未按、按住與放開結果可解釋 |
| 7 | 前六階段完成 | 去抖專題練習 | 能解釋彈跳、區分raw／stable、完成四種設定比較並提出有限制的結論 |

各階段應依序完成。未達成完成條件時，應先依該節故障排除內容修正，再進入
下一階段。


## 二、實驗器材與分組

每個工作站使用一片ESP32-S3、一條可傳輸資料的USB線、一片400孔麵包板、
一顆四腳按鈕、至少四條公對公杜邦線，以及一台筆電。萬用電表由課堂輪流
提供；馬達、舵機、電池盒及其他高耗電設備不進入工作區。

上課前依[Week 2器材與必帶確認表](#一本週必帶與器材確認)
核對完整型號、數量、取得方式與上電前狀態。


## 三、安全須知

### 接線前先辨認電源與訊號

- **GPIO**（General-Purpose Input/Output，通用輸入／輸出接腳）：由程式讀取
  或控制的訊號腳，不是供應馬達電力的電源腳。GPIO4、GPIO5 是晶片編號，
  不是從板邊數過來的第 4、5 支排針。
- **GND**（Ground）：電路共同的 0V 參考點，也是電流返回路徑。
- **3V3**：板上的 3.3V 電源腳；**5V** 是板上的 5V 電源腳。ESP32-S3 GPIO
  使用 3.3V 邏輯，因此 5V 不可直接進入 GPIO。
- **絲印／pinout**：絲印是板面印的接腳文字；pinout 是整塊板的接腳功能圖。
  接線先核對實物絲印，再核對同一板本的官方 pinout。
- **短路**：兩個不該直接連接的點形成很低電阻的路徑，例如 5V 直接接 GND，
  可能造成過大電流、發熱或損壞。

1. **接線或改線前，先拔除 USB。**
2. 5V 不得接入任何 GPIO。
3. GPIO 是訊號腳，不得直接作為舵機或馬達的電源。
4. 接腳不確定時，應核對板身絲印與官方 pinout，不得憑記憶接線。
5. 接回 USB 前，必須依接線表逐條檢查。身邊有同學時，請同學和你一起檢查；
   獨自操作時，先從 ESP32 腳位沿線檢查到零件，再從零件反向檢查回 ESP32。
6. 發現板子、線材或零件發熱、異味或異常聲音，立即斷電並通知教師。

教師會先以GPIO4與GPIO5作為候選profile進行target test。YD板實物絲印列出兩個
腳位，Espressif晶片與模組資料可用來查核其限制，但不能用DevKitC-1的PCB外觀或
header位置取代本批YD板target test。教師公布
profile後，學生須核對profile、板身絲印與金屬屏蔽罩。所有後續接線都以profile中的
`PIN_BUTTON`與`PIN_TEST_OUTPUT`為準，不依候選值猜測。為避免記憶體配置差異，本課不使用GPIO35、GPIO36、GPIO37，
因為使用 Octal SPI Flash／PSRAM 的相關版本會把這些腳位保留給板內部通訊。

板載 RGB LED（可顯示紅、綠、藍的多色燈）不列入本週必要任務。不同開發板
可能把它接到不同GPIO；未完成YD板實測前，不把網路上的
LED 腳位直接套用到實物。

官方DevKitC-1照片只能用來比較「同一晶片的不同開發板」；本課YD板的USB與按鈕
位置以實物為準。晶片與模組規格另見
[ESP32-S3-WROOM-1資料表](https://documentation.espressif.com/esp32-s3-wroom-1_wroom-1u_datasheet_en.pdf)。


## 四、開發環境與板卡辨識

### 步驟 1：安裝或更新 Arduino IDE

#### Arduino IDE

- **Arduino IDE**：撰寫、編譯和上傳 ESP32 程式的軟體。
- **stable version／穩定版**：Arduino 正式提供給一般使用者的版本。本課使用
  官方穩定版，不使用 Nightly Builds 測試版。

若電腦尚未安裝 Arduino IDE，或版本不是目前課程採用的版本，依下列順序操作：

1. 用瀏覽器開啟 [Arduino Software 官方下載頁](https://www.arduino.cc/en/software)。
2. 找到 **Arduino IDE 2**，選擇適合目前 Windows 的 64-bit 安裝版本。不要下載
   `Arduino IDE 1.8.x` 或 `Nightly Builds`。
3. 下載頁若出現捐款選項，可按 **JUST DOWNLOAD** 直接下載。
4. 開啟下載的安裝程式。Windows 顯示使用者帳戶控制確認時，先核對發布者與
   下載來源，再允許安裝。
5. 保留預設安裝元件與安裝位置，完成安裝後啟動 Arduino IDE。
6. 在 Arduino IDE 點 **Help → About Arduino IDE**，記錄 `Version`。關閉
   About 視窗後才繼續下一步。

若電腦原本已有 Arduino IDE，也必須以 **Help → About Arduino IDE** 讀取實際
版本；桌面圖示存在並不能證明版本正確。安裝或更新 Arduino IDE 時不需要連接
ESP32。

正常結果：Arduino IDE 主視窗可以開啟，About 視窗顯示 Arduino IDE 2 的版本。
若安裝程式無法啟動，先保存完整 Windows 訊息並確認檔案來自 Arduino 官方網站，
不要改從來路不明的網站下載。

#### 步驟 2：安裝或驗證 ESP32 開發板支援套件

- **Boards Manager**：Arduino IDE 內安裝開發板支援套件的位置。
- **package／platform**：使 Arduino IDE 認得 ESP32、提供編譯工具與板型設定
  的軟體套件。本課選作者為 Espressif Systems 的 `esp32`。
- **repository／repo**：由 Git 管理的課程專案資料夾；GitHub 上看到新版不
  代表本機一定已同步。

依下列順序安裝或確認，不得只看到名稱包含 ESP32 就選取：

1. 啟動 Arduino IDE 2，等待主視窗完整顯示。
2. 點左側 **Boards Manager** 圖示；若看不到，使用選單
   **Tools → Board → Boards Manager**。
3. 在搜尋欄輸入 `esp32`。
4. 找到作者為 **Espressif Systems** 的 `esp32` package。
5. 若按鈕顯示 **INSTALL**，按下後等待下載與安裝完成。安裝期間保持網路連線，
   不要關閉 Arduino IDE。
6. 安裝完成後，確認卡片顯示 **REMOVE** 或明確標示已安裝版本。`REMOVE` 是
   移除，不是完成按鈕；已安裝時不要按下它。
7. 把實際版本填入：

```text
Arduino IDE 版本：____________________
Espressif esp32 package 版本：____________________
```

正常結果：搜尋結果顯示作者為 **Espressif Systems** 的 `esp32`，並能讀到已安裝
版本。若下載中斷，先保存紅色錯誤訊息；不要同時改網路、重裝 IDE 與安裝未知
driver，否則無法判斷是哪一項修正有效。

#### 步驟 3：確認課程 repository

1. 用瀏覽器開啟本 repository，確認目前看到的檔名是 `week2_main.ipynb`。
2. 若使用本機 Git，先確認本機分支已與課程 GitHub 同步；看到 GitHub 新版不代表
   本機檔案已自動更新。

最後勾選：

- [ ] Arduino IDE 2 可以正常開啟。
- [ ] Boards Manager 顯示 Espressif `esp32` platform 已安裝。
- [ ] 已取得最新版課程 repository。
- [ ] USB 線已知具有資料傳輸功能。

任一項未完成時，應先進入環境排錯區。每位學生均須完成自己的環境驗收。

### 步驟 4：辨識板卡，不接線

#### 板卡組成

- **PCB**（Printed Circuit Board，印刷電路板）是黑色長條板本身。板內的銅線
  連接USB、按鈕、模組、電源與兩排接腳；板面印出的白色名稱稱為**絲印**。
- **ESP32-S3** 是執行程式、讀取接腳及控制輸出的微控制器晶片。
- **ESP32-S3-WROOM-1 N16R8模組**是左側帶金屬屏蔽罩的長方形組件，包含
  ESP32-S3、16 MB Flash、8 MB PSRAM及PCB天線。
- **YD-ESP32-S3 Type-A V1.5開發板**是整塊PCB與焊在上面的模組、USB接頭、
  CH343、穩壓與保護元件、按鈕、指示燈及排針。晶片、模組與開發板是三個
  不同層次，不能把其中一個名稱當成另外兩個。

```text
YD-ESP32-S3 Type-A V1.5開發板
├─ 黑色PCB、USB接頭、按鈕、CH343與排針
└─ ESP32-S3-WROOM-1 N16R8模組
   ├─ ESP32-S3晶片
   ├─ 16 MB Flash
   ├─ 8 MB PSRAM
   └─ PCB天線
```

下圖由本課實物正面照片製作，只協助辨認外形與位置。後製標線不能取代板身
絲印、官方文件、通斷量測或target test。

![YD-ESP32-S3 Type-A V1.5主要元件辨識圖](../../docs/images/hardware/guides/yd-esp32-s3-front-annotated-components-v2.png)

#### RST與BOOT不是相同按鈕

| 按鈕 | 目前用途 | 操作後發生什麼事 |
|---|---|---|
| `RESET`／`RST` | 重新啟動ESP32 | 停止目前執行，從Flash重新載入原程式，再從`setup()`開始；不會刪除程式 |
| `BOOT` | 在啟動時選擇下載模式 | 通常配合RST使用；單獨按下不等於重新啟動，也不會自行上傳程式 |

正常Upload由CH343與控制訊號自動進入下載及Reset流程，不需要先按任何按鈕。
只有Upload停在`Connecting...`且完成基本排查後，才使用本章後面的手動
BOOT／RST程序。短按RST後，`millis()`與`uptime_ms`會重新從接近0開始，
因此可用Serial輸出確認程式是否真的重新啟動。

#### 排針旁的文字分成電源、控制與GPIO

- **GPIO**（General-Purpose Input/Output，通用輸入／輸出接腳）是由程式
  讀取或控制的訊號腳。板上的`4`表示GPIO4，`5`表示GPIO5；數字不是從板邊
  數來的第4、5支腳，也不是電壓。
- **GND**是0V共同參考與電流返回路徑。HIGH／LOW電壓都必須相對GND判讀。

#### GND與GPIO的差異

| 接腳 | 固定或可程式控制 | 本週作用 | 例子 |
|---|---|---|---|
| `GND` | 固定作為0V參考，不由程式切換 | 提供比較電壓的基準及電流返回路徑 | 萬用電表黑表筆接GND後，紅表筆量到的電壓才有共同基準 |
| `GPIO` | 可由程式設為輸入或輸出 | 輸入時讀取HIGH／LOW；輸出時產生受控制的HIGH／LOW | GPIO4可讀按鈕，GPIO5可作測試輸出 |

可把GND想成量高度時的地面0公尺：它不負責傳遞『按下』或『放開』的資料，但沒有0公尺基準，就無法說另一個位置有多高。GPIO則像相對地面改變或被量測的位置；GPIO的HIGH或LOW都必須相對GND才有意義。

本週按鈕範例會把候選GPIO4設為`INPUT_PULLUP`輸入。放開按鈕時，晶片內部的上拉電阻使GPIO4讀到HIGH；按住按鈕時，按鈕把GPIO4接到GND，因此讀到LOW。GND沒有『傳送LOW命令』，而是提供0V狀態；真正被程式讀取的是GPIO4。候選GPIO5則作為輸出測試點：程式輸出HIGH時，它相對GND應接近3.3V；輸出LOW時應接近0V。

#### 上拉電阻是什麼？3.3V從哪裡來？

`pull-up resistor`（上拉電阻）是在沒有其他裝置主動控制輸入時，透過電阻把輸入接向較高電位，使它有明確預設狀態的元件。`pull`表示把訊號維持在某個方向，`up`在本例表示朝3.3V的HIGH方向。它不會自己產生3.3V，只是把既有的3.3V電源經過電阻連到GPIO4。

本次使用ESP32-S3內部上拉，不需要從電阻包另外拿一顆。程式執行

```cpp
pinMode(PIN_BUTTON, INPUT_PULLUP);
```

後，晶片才啟用GPIO4內部通往3.3V電源軌的弱上拉路徑。`INPUT`表示程式讀取這支接腳，`PULLUP`表示同時啟用內部上拉。內部上拉的實際阻值會受晶片與工作條件影響，本課不把某個單一阻值當成固定規格。

本次以`COM` USB-C接頭供電時，電源路徑可先理解為：

```text
電腦USB約5V
  → USB資料線與COM USB-C接頭
  → 開發板上的穩壓電路
  → 約3.3V電源軌
  → ESP32-S3與內部上拉電阻
```

開發板上的穩壓電路把USB提供的約5V轉為ESP32-S3使用的約3.3V；板邊的`3V3`接腳也連到這條3.3V電源軌。這是標稱工作關係，稍後仍要用萬用電表量測本片實物。

> 常見誤解：不是『電腦USB提供3.3V，再轉成3.3V』。本次路徑是『電腦USB提供約5V，開發板穩壓成約3.3V』。穩壓是開發板通電後由硬體自動完成，不必等待Arduino程式執行。USB線同時具有電源線與資料線；本板的`COM` USB-C接頭可供電，並透過CH343處理電腦與ESP32之間的序列通訊。

#### 3.3V電源與GPIO4、GPIO5有什麼關係？

3.3V電源軌先供應ESP32-S3內部處理器、記憶體與GPIO控制電路工作。GPIO則是晶片連到外部世界的訊號接腳。接上USB只會讓晶片取得工作電源，不會使每支GPIO自動固定輸出3.3V；每支GPIO的作用仍由程式設定。

| 本週接腳 | 程式設定 | 與3.3V的關係 |
|---|---|---|
| 候選GPIO4 | `INPUT_PULLUP`輸入 | 透過晶片內部上拉電阻弱連到3.3V，按鈕放開時讀到HIGH；按住時由按鈕接到GND而讀到LOW |
| 候選GPIO5 | `OUTPUT`輸出 | `digitalWrite(..., HIGH)`時由內部輸出電路使接腳接近3.3V；`LOW`時使接腳接近GND的0V |
| GND | 固定參考與返回端 | 不由程式改變，提供GPIO4、GPIO5共同的0V基準 |

可把GPIO5的內部輸出電路想成由程式控制的選擇器：

```text
程式要求HIGH：3.3V電源軌 ──[內部輸出電路]── GPIO5 ≈ 3.3V
程式要求LOW ：GND 0V      ──[內部輸出電路]── GPIO5 ≈ 0V
```

這只是功能示意，不表示GPIO5適合當作一般電源輸出。GPIO只能處理有限電流，本週只讓GPIO5接電表這種高阻抗量測，不用它直接供應舵機、馬達或其他高電流負載。HIGH與LOW的實際電壓稍後必須相對GND量測，不能只由程式文字推定。

目前板卡完全斷電，因此現在並沒有可供上拉使用的3.3V，`INPUT_PULLUP`也尚未執行；此處先解釋接USB並執行程式後會形成的狀態。

#### 什麼是封閉迴路？

`closed circuit`（閉合電路）或此處所說的封閉迴路，是一條沒有中斷、能讓電流從電源的一端出發，經過導線與元件，再返回電源另一端的完整路徑。它不必在外形上畫成圓圈；重點是各接點在電氣上首尾相接。`open circuit`（開路）則表示其中一處中斷，正常工作電流無法沿預定路徑持續流動。

手電筒是直接例子：電池、導線、燈泡與閉合的開關形成完整路徑時，燈泡會亮；開關打開後，路徑被切斷，電池仍有電壓，但燈泡沒有正常工作電流。按鈕的四支腳也是同一概念：放開時兩組接點之間為開路，按住時兩組接點閉合。

封閉迴路不等於短路。正常封閉迴路中有燈泡、電阻、晶片輸入或其他負載來限制或使用電流；短路則是用極低阻抗路徑直接跨接不同電位，例如把3.3V直接接到GND。兩者都有完整路徑，但短路的電流可能過大。

按鈕放開時：

```text
3.3V ──[內部上拉電阻]── GPIO4 ──[按鈕開路]── GND
                          │
                       讀到HIGH
```

按鈕按住時：

```text
3.3V ──[內部上拉電阻]── GPIO4 ──[按鈕閉合]── GND
                          │
                       接近0V，讀到LOW
```

上拉電阻讓按鈕放開時不會懸空，也在按下時限制從3.3V流向GND的電流。假設只為說明而使用10 kΩ電阻，歐姆定律會得到`3.3V ÷ 10000Ω = 0.00033A`，也就是0.33mA；這個計算示範電阻如何限制電流，不代表ESP32-S3內部上拉一定是10 kΩ。若完全沒有這個電阻而直接把3.3V接到GND，就會形成不安全的低阻抗路徑。

#### GND不傳資料，為什麼仍然必須接？

電壓是兩個位置之間的差，不是單一接腳自己擁有的數字。量測GPIO5時，紅表筆碰GPIO5、黑表筆碰GND，電表顯示的是`GPIO5電位－GND電位`。若不接GND，電表沒有共同的0V基準，讀值就不能代表GPIO5相對本電路的HIGH或LOW。

按鈕也需要GND。按住按鈕時，預期路徑是`晶片內部3.3V → 上拉電阻 → GPIO4 →按鈕 → GND`。上拉電阻限制電流，GND提供返回路徑，GPIO4才會接近0V並被讀成LOW。如果移除GND線，按鈕只會把GPIO4接到一個懸空接點；按下動作不再提供可靠LOW，程式可能一直讀到HIGH或得到不穩定結果。因此『GND不承載按鈕資料』不等於『GND可以省略』。

#### 沒有GND是否代表電流一定不會流動？

更精確的判斷是：電流要持續流動，必須有從電源出發並返回電源的封閉路徑。手電筒不一定在外殼上寫`GND`，但電池正極、燈泡與電池負極形成封閉回路時，電流仍可流動；電池負極在該電路中扮演返回端與參考點。若拆開其中一條線，電池兩端仍可能存在電壓，但燈泡的正常工作電流無法持續流過。這說明『有電壓』和『有完整電流路徑』是兩件事。

在本次ESP32按鈕實驗中，開發板內部原本就有GND，USB供電時也會把電源返回端接入開發板；棕色線的工作是把麵包板上的按鈕接回同一個GND。若棕色線未接，外部按鈕缺少預期返回路徑，GPIO4成為懸空或維持上拉HIGH。實際電路仍可能存在極小漏電流、人體或環境造成的雜訊路徑，所以不要把它說成所有情況下電流絕對為零；正確結論是沒有可靠、可預測的工作回路。

BOARD-T01此次實作使用棕色線接GND、紅色線接候選GPIO4、橘色線接候選GPIO5。線色只是方便追蹤的標籤，不會讓導線自動成為GND或GPIO；真正用途由它接到的板身絲印位置決定。其他學生可使用不同顏色，但必須先記錄對應關係。

不要把設定為HIGH的GPIO輸出直接用導線接到GND。這相當於把約3.3V輸出直接接到0V，可能形成過大電流並損壞接腳。按鈕接到GPIO輸入能安全使用，是因為`INPUT_PULLUP`路徑內含限制電流的上拉電阻，而且該GPIO此時不是主動輸出HIGH。

#### 為什麼板上看得到兩個GND，但本次只接一條GND線？

開發板會把同一個GND網路引到多支排針，讓不同方向的線路都能方便取得0V參考。這些GND接腳不是不同種類的負電，也不會因為位於板子的上方或下方而改變作用。在本批BOARD-T01正面朝上、天線朝上且USB朝下時，右排上方與右排下方都可看到`GND`絲印，但本次三條公對母杜邦線的用途固定為一條GND、一條GPIO4及一條GPIO5，不需要同時連接兩個GND。

本次選右排最上方的GND，是因為它容易與下方緊鄰`5Vin`和USB的電源區分開，也能讓三條線較容易整理；右排下方的GND留空。兩個GND是否相通可另在完全斷電時進行通斷測試，但那是額外的板卡辨識驗證，不是目前三線接法的一部分。
- **3V3**是3.3V電源軌，不等於一般GPIO。GPIO輸出HIGH可能接近3.3V，仍只應
  視為控制訊號，不拿來替高電流負載供電。
- **5Vin**讀作「five-volt input」，是5V電源軌標示，不是`5Yin`，也不是GPIO。
  本週不使用5Vin；未確認電源路徑及防回灌設計前，不同時從USB與外部電源供電。
- **TX**與**RX**是序列傳送與接收訊號，`RST`是重設線；它們也不是一般電源腳。

ESP32-S3 GPIO使用3.3V邏輯，5V不得直接進入GPIO。不同編號可能另有開機、USB、
UART、Flash或PSRAM功能，不能因為板上印有數字就任意使用。

拿起 ESP32-S3，依板身標示找出：

- 兩個 USB 接頭及其絲印。
- BOOT 按鈕。
- RESET／RST 按鈕。
- 3V3、5V、G／GND。
- 教師profile所列的`PIN_BUTTON`與`PIN_TEST_OUTPUT`絲印位置。

若使用筆電相機保存板卡辨識證據，依
[Windows相機拍攝與板卡照片檢核](#使用-windows-相機拍攝與板卡照片檢核)
拍攝。拍照時板卡不得連接USB、電池或其他線路。

本批板卡不是官方DevKitC-1 PCB，因此不使用DevKitC-1文件中的J1／J3排針位置。
學生應讀教師公布profile，再找YD板上的`4`、`5`、`GND`等實物絲印，不得僅以
排針順序或相似板卡圖片判斷。

BOARD-T01原始照片確認：正面朝上、黑色天線在左、兩個USB接頭在右時，靠近照片
下緣的那排絲印由左起是`3V3`、`3V3`、`RST`、`4`、`5`、`6`……；因此`4`與`5`
是相鄰腳位，分別位於該排由左數第4與第5個位置。相同排最右端印有`GND`，其左側
是`5Vin`。這個計數只用來協助找到文字，最後仍必須看腳位旁的`4`、`5`與`GND`
絲印確認；若學生實物的標示順序不同，立即停止，不得照此位置接線。

依實物填寫下表，不得直接抄錄其他組員的內容：

| 項目 | 板身實際標示／位置 |
|---|---|
| 開發板或模組型號 |  |
| 預計使用的 USB 接頭 |  |
| BOOT |  |
| RESET／RST |  |
| `PIN_BUTTON`（profile值與位置） |  |
| `PIN_TEST_OUTPUT`（profile值與位置） |  |
| GND |  |

### 本節檢核

兩位組員各自在板卡照片上標出 USB、BOOT、RESET、profile指定的兩個GPIO與 GND，
再互相比對；有不同之處就回到板身絲印核對並修正標記。


## 五、Arduino IDE 板卡與連接埠設定

### 步驟 1：建立 USB 連線

- **USB** 在這一步同時供電和傳輸資料；只有充電功能的線可以讓燈亮，卻不
  會讓電腦出現 Port。
- **USB-to-UART** 是把電腦 USB 資料轉成 ESP32 序列通訊的橋接路徑。本週
  使用YD板背面標示`COM`的接頭，不使用背面標示`USB`的原生USB接頭。
- **USB hub** 是把一個 USB 孔擴充成多孔的集線器；鬆動或供電不穩時可能
  造成連線中斷，因此第一次測試先直接接筆電。

1. 將ESP32從包裝泡棉或其他不明底材取下，放在乾燥、不導電且不會滑動的平面；
   確認尚未插入麵包板，也沒有接任何杜邦線。通電時不要用手同時碰觸兩排針腳。
2. 關閉 Arduino IDE 的 Serial Monitor，避免它占用 Port。
3. 斷電時翻到背面，確認兩個接頭旁分別印有`COM`與`USB`。`RST`與`BOOT`在
   正面是並排按鈕，不能用按鈕與接頭是否同列來判斷USB用途。
4. 翻回正面並放平。當天線在左、兩個USB接頭在右時，`COM`是**右上方**、靠近
   `RX`／`TX`／`PWR`指示燈與USB轉序列晶片的接頭；`USB`則是右下方、靠近
   `RGB`區域的原生USB接頭。把資料線接到`COM`的 **USB-to-UART** 接頭。
5. 把另一端接到筆電；不得使用接觸鬆動的 USB hub。
6. 等待作業系統完成裝置辨識。
7. 確認板上電源指示燈亮起。燈亮只證明有電，下一步仍要確認 Port。

此階段僅連接「ESP32、USB 與筆電」，不連接麵包板或其他模組。

本板使用`COM`接頭時，實際資料方向是：

```text
Upload：筆電USB → CH343 → UART RX／TX → ESP32-S3
Serial：ESP32-S3 → UART TX／RX → CH343 → 筆電USB → Serial Monitor
```

CH343是右上方、位於`RX`／`TX`／`PWR`三顆指示燈與上方USB-C接頭之間的
黑色USB-to-UART通訊晶片。它負責轉換通訊格式，不執行Arduino程式；程式由
ESP32-S3執行。背面標示`USB`的原生USB接頭則直接連接ESP32-S3，不經CH343。

### 步驟 2：選擇 Board 與 N16R8 設定

- **Board** 是 Arduino IDE 的編譯目標設定，告訴工具要為哪種晶片與硬體
  產生程式；它不是 USB Port。
- **N16R8** 是模組容量標示：`N16` 表示 16 MB Flash，`R8` 表示 8 MB
  PSRAM；必須先在金屬屏蔽罩看到相同標示，才能使用後面的候選設定。
- **Flash** 是斷電後仍保存程式的快閃記憶體；**PSRAM**（Pseudo Static RAM）
  是程式執行時使用、斷電後不保留內容的額外記憶體。
- **QIO**（Quad I/O，四線輸入／輸出）與 **OPI**（Octal Peripheral
  Interface，八線周邊介面）是記憶體資料傳輸模式。WROOM-1 N16R8 的 Flash
  使用 QIO，PSRAM 使用 OPI；不是看到較大的數字就任意選最快設定。
- **Flash Mode／Flash Size** 分別指定 Flash 的通訊方式與容量，兩者都必須
  符合實物 N16 規格。
- `MB` 的大寫 `B` 表示 byte，`Mb` 的小寫 `b` 表示 bit；8 bits = 1 byte，
  所以 128 Mb 等於 16 MB。
- **USB CDC** 是讓原生 USB 表現成序列通訊埠的功能；本週經 USB-to-UART，
  所以 `USB CDC On Boot` 選 Disabled。
- **UART**（Universal Asynchronous Receiver/Transmitter）是序列通訊硬體；
  UART0 是 ESP32 的其中一組通道。本週 Upload Mode 使用表中指定的
  `UART0 / Hardware CDC`。
- **Upload** 是把程式寫入 ESP32；Upload Mode 選通訊路徑，Upload Speed 選
  傳輸速度。真正執行 Upload 會在下一階段操作。
- **Partition Scheme** 決定 Flash 如何分配給程式和其他資料；**Erase All
  Flash** 則決定上傳前是否清除整顆 Flash。本週只照表設定，不自行改動。
- **Sketch** 是 Arduino 對一個程式專案的稱呼，因此選單中的 `Before Sketch
  Upload` 就是「上傳程式前」。

本課選擇的`ESP32S3 Dev Module`是Arduino IDE中的通用編譯profile，不是實物
PCB名稱。四個名稱的關係如下：

```text
Arduino IDE Board設定：ESP32S3 Dev Module
實際開發板PCB：YD-ESP32-S3 Type-A V1.5
實際模組：ESP32-S3-WROOM-1 N16R8
程式讀到的晶片：ESP32-S3
```

選對通用Board只代表工具使用ESP32-S3的編譯規則，不能因此推定USB接頭位置、
固定GPIO或整塊PCB就是Espressif原廠DevKitC-1。

使用教師標記的 USB-to-UART 接頭時，N16R8 規格對應的候選設定如下。
Arduino-ESP32 版本不同時，選單文字與預設值可能不同；教師必須先在本批
實物完成 Upload、Serial 與重新開機測試，並把設定截圖記錄到
`docs/hardware_state.md`。學生以教師公布的實機驗證截圖為準；尚未公布時
不得猜設定或直接上傳。

| 設定 | N16R8 候選值／授課時使用方式 |
|---|---|
| Board | `ESP32S3 Dev Module` |
| Flash Mode | `QIO 80MHz`，若選單分開則 Flash Mode 選 QIO |
| Flash Size | `16MB (128Mb)` |
| PSRAM | `OPI PSRAM` |
| USB CDC On Boot | `Disabled`（本週使用 USB-to-UART） |
| Upload Mode | `UART0 / Hardware CDC` |
| Upload Speed | 先用預設；不穩定時降低一級再試 |
| Partition Scheme | 使用教師在相同 package 版本完成實機驗證的值，不自行挑選 |
| Erase All Flash Before Sketch Upload | `Disabled` |

Espressif 的 WROOM-1 模組資料表列出 N16R8 為 16 MB Quad SPI Flash 與
8 MB Octal SPI PSRAM；Arduino-ESP32 工具說明則要求設定符合實際模組。
這只能支持 QIO、16 MB 與 OPI 的規格判斷，不能取代本批開發板、USB 路徑、
  package 版本與 Partition Scheme 的指定實機測試（target test，也就是使用
  課堂實際板卡、線材與軟體版本完成操作）。若金屬屏蔽罩實際不是
`ESP32-S3-WROOM-1 N16R8`，立即停止並請教師重新核對。

官方參考：[ESP32-S3-WROOM-1 模組資料表](https://documentation.espressif.com/esp32-s3-wroom-1_wroom-1u_datasheet_en.pdf)｜[Arduino-ESP32 Tools Menu](https://docs.espressif.com/projects/arduino-esp32/en/latest/guides/tools_menu.html)｜[PSRAM 設定排錯](https://docs.espressif.com/projects/arduino-esp32/en/latest/troubleshooting.html)

實際點選順序：

1. 點 **Tools → Board → esp32 → ESP32S3 Dev Module**。
2. 再打開 **Tools**，逐項找到上表選項。
3. 每設定一項後回到 Tools 設定下一項；選擇 Board 不會自動完成其餘設定。
4. 對照教師公布、且已記錄 package 版本的實機驗證截圖；任一值不同時先停止，
   不自行判斷哪一個比較快或比較新。
5. 設完後重新打開 Tools，由上往下逐項比對一次。
6. 把 Tools 選單截圖保存為 `week02_board_settings_組別.png`。

如果 Tools 中完全看不到 ESP32-S3、Flash Size 或 PSRAM，通常是選錯 Board，
或 Espressif `esp32` package 沒有正確安裝。回到階段 1，不要繼續 Upload。

### 步驟 3：選擇 Port

- **Port** 是 Arduino IDE 要和哪一個已連接裝置通訊。Windows 常顯示為
  `COM5`、`COM6` 等，每台電腦和每次插孔都可能不同。
- 本節的 **COM5** 是 Windows 序列連接埠名稱；後面萬用電表上的 **COM**
  是黑表筆插孔，兩者完全不同。
- **driver** 是讓 Windows 辨認 USB 裝置並建立 Port 的驅動程式，不應看到
  問題就隨機安裝來源不明的 driver。

用「拔除前後比較」找 Port：

1. 先拔掉 ESP32 的 USB。
2. 打開 **Tools → Port**，把目前清單記下來或截圖。
3. 關閉 Port 選單。
4. 把 ESP32 接回同一個 USB 孔，等待裝置辨識。
5. 再開 **Tools → Port**。
6. 找出新出現的 Port，例如 Windows 的 `COM5`。
7. 點選該 Port；被選取的項目前應出現勾選符號。
8. COM 號碼由各台電腦分配，不得直接套用其他組別的號碼。

Windows可能原本就列出多個`Bluetooth`／「透過藍牙連結的標準序列」COM Port。
它們不是ESP32。這批YD板的`COM`接頭在本次測試顯示為
`USB-Enhanced-SERIAL CH343 (COM8)`；`CH343`是板上USB-to-UART橋接晶片的名稱，
`COM8`則只是這台電腦當次分配的號碼。學生仍須用拔除前後比較找出自己新增的
`USB-Enhanced-SERIAL CH343`，不能照抄別人的COM號碼，也不能選Bluetooth Port。

把實際 Port 寫下來：

```text
我的 Port：____________________
```

如果沒有新 port：

1. 先換成已知可傳資料的 USB 線。
2. 換另一個電腦 USB 接頭。
3. 關閉並重新開啟 Arduino IDE 的 port 選單。
4. 查看 Windows 裝置管理員是否出現未知裝置。
5. 保存畫面後交由教師協助，不得安裝來源不明的 driver。

### 本節檢核

- [ ] Board 是 `ESP32S3 Dev Module`。
- [ ] Flash Size 是 16MB，PSRAM 是 OPI。
- [ ] 使用 USB-to-UART 時，USB CDC On Boot 是 Disabled。
- [ ] 已用拔除前後比較找到自己的 Port。
- [ ] Board 設定截圖已保存。


## 六、第一個程式：編譯、上傳與序列輸出

### 步驟 1：建立程式

- **Sketch** 是 Arduino 對一個程式專案的稱呼，資料夾和主要 `.ino` 檔通常
  使用同一名稱。
- **`setup()`** 在開機或 RESET 後只執行一次；**`loop()`** 會在之後持續
  重複執行。
- `CHANGE_ME` 和 `XX` 是「請換成自己的資料」的預留文字，不是固定答案。
- **baud rate** 是序列通訊速度。本課程式使用 115200，稍後 Serial Monitor
  也必須選相同數值。

1. 點 **File → New Sketch**。
2. 點 **File → Save As**。
3. 將資料夾／Sketch 名稱設為 `week02_serial_groupXX`，把 `XX` 換成組別。
4. 刪除編輯器內原本的 `setup()` 與 `loop()` 範本，避免重複定義。
5. 使用 GitHub 程式區塊右上角的 Copy 按鈕，完整複製下列程式。
6. 貼到 Arduino IDE。
7. 把 `CHANGE_ME` 改成組別，例如第 3 組改成 `03`。
8. 按 **Ctrl+S** 儲存。

#### 程式語法說明

- `unsigned long` 是可保存非負整數的資料型別，本程式用來保存毫秒時間。
- `millis()` 取得 ESP32 從開機到目前經過的毫秒數。
- `if` 表示只有括號內條件成立，才執行大括號內的程式；`>= 1000` 表示已經
  過了至少 1000 毫秒。
- `Serial.begin(115200)` 以 115200 baud 啟動序列通訊。
- `Serial.println()` 輸出一整行文字；`Serial.printf()` 可把變數插入文字，
  `%lu` 對應 `unsigned long`，`\n` 表示換行。

完整程式：


In [ ]:
unsigned long lastReportMs = 0;

void setup() {
  Serial.begin(115200);
  delay(500);
  Serial.println("boot: week02 group=CHANGE_ME version=1");
}

void loop() {
  unsigned long now = millis();

  if (now - lastReportMs >= 1000) {
    lastReportMs = now;
    Serial.printf("status: uptime_ms=%lu\n", now);
  }
}


baud rate 維持 115200。貼上後確認程式只出現一組 `setup()` 與一組 `loop()`。

### 步驟 2：分辨 Verify 與 Upload

- **Verify／Compile（驗證／編譯）**：檢查語法，再把人寫的程式轉成 ESP32
  可執行的韌體；成功不代表已經寫進板子。
- **韌體（firmware）**：儲存在 ESP32 Flash、開機後會執行的程式。
- **Upload（上傳／燒錄）**：把編譯完成的韌體寫入板子。
- **Output**：IDE 下方顯示編譯和上傳過程的區域，不是稍後顯示程式文字的
  Serial Monitor。

1. 點左上角勾號 **Verify**。
2. 檢查視窗下方 Output 的文字，不以進度動畫作為成功判斷。
3. 等待編譯完成及記憶體用量訊息；若出現紅色錯誤，應先處理 Output 中最早
   出現的錯誤，不從最後一行反向推測。
4. Verify 成功後，點右箭頭 **Upload**。
5. Output 會先再次編譯，再出現連線、寫入百分比及完成訊息。
6. 等待 `Hard resetting via RTS pin...` 或 IDE 顯示 Upload 完成。
7. Upload 期間不得拔線、按 RESET 或移動板子。

`Hard resetting via RTS pin...` 通常表示 IDE 已透過 USB-to-UART 的 RTS 控制
訊號自動重設板子，讓新程式開始執行；它不是「硬體損壞」訊息。

Verify成功時常見的記憶體摘要可以這樣判讀：

```text
Sketch uses ... bytes (...) of program storage space.
Maximum is ... bytes.
Global variables use ... bytes (...) of dynamic memory.
```

- `Sketch uses`是本次韌體占用的程式分割區空間。
- `Maximum`是目前Partition Scheme分給應用程式的上限，不是整顆Flash容量；
  16 MB Flash中的其他區域可能分給檔案系統、系統資料或其他映像。
- `Global variables`是編譯時可計算的內部記憶體配置，不能用來證明8 MB PSRAM
  已經啟用；PSRAM仍須由程式執行期間的runtime讀值確認。

Upload時的`Writing at ... 100%`表示資料已傳送到Flash；`Hash of data verified`
表示寫入後資料與本次傳送內容的雜湊比對相符。這些訊息證明寫入流程完成，
仍不能證明程式功能、GPIO或外接硬體正確。`Sketch uses`與`Wrote`的bytes可能因
映像區段、對齊或傳輸格式而不同，不必強求兩個數字相同。

Verify 成功不等於 Upload 成功；Upload 成功也不等於程式功能正確。三者需要
不同證據。

#### `Connecting...` 停滯時的處理

`Connecting...` 表示 IDE 正嘗試和 ESP32 建立下載連線。BOOT 可讓板子進入
下載模式，RESET 讓板子重新啟動；只有正常自動 Upload 失敗時才使用下列
手動順序。

依下列順序做一次手動下載模式：

1. 保持 USB 連接。
2. 按住板上的 **BOOT** 不放。
3. 短按一下 **RESET／RST** 後放開 RESET。
4. 再放開 BOOT。
5. 回到 Arduino IDE，重新確認 Port。
6. 再按 Upload。

另一種常見操作是先按 Upload，看到 `Connecting...` 時按住 BOOT，連線開始
寫入後再放開。兩種方式都只在正常自動 Upload 失敗時使用。

若 Output 顯示晶片不是 ESP32-S3，應立即停止並回到 Board 設定，不得使用
錯誤 Board 強行上傳。

### 步驟 3：檢查 Serial Monitor 輸出

- **Serial** 是 ESP32 與電腦依序傳送文字或資料的通訊方式。
- **Serial Monitor** 是 Arduino IDE 顯示這些程式輸出的視窗。
- **log** 是程式留下的執行紀錄，例如組別、版本、事件與時間。
- **115200 baud** 是本程式的傳輸速度；程式與 Serial Monitor 設定不同時，
  文字可能變成亂碼。

Serial Monitor不會自動掃描晶片或顯示「所有硬體參數」。它只顯示程式透過
`Serial.print()`、`Serial.println()`或`Serial.printf()`主動送出的內容。
`Serial.begin(115200)`設定ESP32送出序列資料的速度，Monitor也必須選115200；
Upload Speed即使也是115200，仍屬於「電腦把韌體送到板子」的另一個階段。

1. Upload 完成後，點 Arduino IDE 右上角 **Serial Monitor** 圖示，或選
   **Tools → Serial Monitor**。
2. 在 Serial Monitor 的 baud rate 選單選擇 `115200`。
3. 按一下板上的 RESET，讓開機訊息重新出現。
4. Serial 輸入框保持空白；本程式不讀取鍵盤輸入。
5. 預期看到：

```text
boot: week02 group=03 version=1
status: uptime_ms=1000
status: uptime_ms=2000
status: uptime_ms=3000
```

`boot: ...`是程式自己寫出的開機訊息，不是ESP32內建健康報告。`uptime_ms`來自
`millis()`，表示本次開機或Reset後經過的毫秒數；增加約1000代表約一秒。按RST
後再次看到`boot: ...`，且uptime重新從接近0開始，才形成「程式已重新啟動」的
可觀察證據。若Serial Monitor在`setup()`完成後才開啟，可能先只看到uptime；
短按RST可以重現開機訊息。Monitor保留的Reset前舊文字不代表兩份程式同時執行。

若是亂碼：

1. 確認 baud rate 是 115200。
2. 確認 Serial Monitor 使用的 Port 和 Upload 的 Port 相同。
3. 按 RESET 再看一次。

若完全沒有文字：

1. 關閉 Serial Monitor。
2. 重新確認 Tools → Port。
3. 再開 Serial Monitor 並選 115200。
4. 按 RESET。
5. 檢查程式是否包含 `Serial.begin(115200)`。
6. 仍無輸出時重新 Upload，保存 Output 及 Serial 畫面再求助。

### 步驟 4：驗證程式版本

1. 關閉 Serial Monitor。
2. 在程式中把 `version=1` 改成 `version=2`。
3. 按 Ctrl+S。
4. 再按 Upload。
5. Upload 完成後重新開啟 Serial Monitor，確認 115200。
6. 按 RESET。
7. Serial 必須出現正確組別及 `version=2`。
8. 保存包含組別、版本與 uptime 的畫面，命名
   `week02_serial_version2_組別.png`。

### 本節檢核

- [ ] Verify 成功。
- [ ] Upload 成功。
- [ ] Serial 顯示正確組別與 `version=2`。
- [ ] 實驗紀錄已各用一句文字記錄 Verify 與 Upload 的用途。

BOARD-T01的實際Compile、Upload、Hash、Reset、Flash、PSRAM與Serial證據逐項
解讀見[本notebook附錄的基本驗證案例](#八board-t01基本驗證案例如何判讀)。


## 七、麵包板與按鈕接線

### 步驟 1：斷開 USB 電源

1. 關閉 Serial Monitor。
2. 從 ESP32 端拔除 USB 線。
3. 確認板上電源燈熄滅。
4. 等待數秒後，再拿出麵包板、按鈕與杜邦線。

後面凡是寫「拔除 USB」，都要做到電源燈熄滅。只關閉 Serial Monitor 或只
停止程式，都不等於斷電。

### 步驟 2：確認麵包板導通結構

**麵包板（solderless breadboard）**是不必焊接就能暫時接線的實驗板。方孔
只是讓導線或元件腳進入；真正建立連接的是塑膠外殼下方的金屬彈片。新麵包板
第一次插線可能較緊，應捏住靠近金屬針的黑色接頭，讓公針垂直進孔並穩定下壓。
公針若開始彎曲就停止，不使用鉗子硬壓，也不把較粗的萬用電表表筆插進孔內。

本課實物是有`a～j`及`1～30`座標、左右各有紅藍電源軌的400孔麵包板：

![400孔麵包板實物俯視圖](../../docs/images/hardware/actual/breadboard-400-tie-point-actual-top.jpg)

本課杜邦線實物包含公對公、公對母與母對母。插入麵包板時要使用有裸露金屬
公針的一端；顏色只協助整理線路，不會改變電氣功能。

![三種杜邦線實物](../../docs/images/hardware/actual/jumper-wires-assorted-actual.jpg)

#### 中央實驗區的座標與五孔組

把麵包板直放、列號由上往下增加時，同一列可畫成：

```text
a10—b10—c10—d10—e10   │   f10—g10—h10—i10—j10
       左側五孔組       │          右側五孔組
                        │
                     中央溝槽
```

- `a10～e10`由同一片金屬彈片連接。
- `f10～j10`是另一片獨立金屬彈片。
- `e10`與`f10`雖然列號相同，仍被中央溝槽隔開。
- `a10`與`a11`雖然上下相鄰，仍屬不同列，不會自動連接。

這表示插在同一五孔組中的零件腳在電氣上屬於同一個**節點（node）**。例如兩支
零件腳同時插在`a10`與`b10`，並不是形成兩段串聯電路，而是把兩支腳直接接在
同一節點。

#### 斷電實測麵包板內部結構

下列測試使用兩條公對公杜邦線延伸測點。ESP32、USB及電池必須全部斷開；黑表筆
留在`COM`、紅表筆留在`VΩmA`，旋鈕使用聲波符號的通斷檔。先短接兩支表筆確認
會蜂鳴，再將杜邦線公針垂直插入指定孔，表筆只碰另一端外露的金屬針。

| 實際測點 | 2026-08-29實測 | 可以支持的判斷 |
|---|---|---|
| `a10`與`b10` | 有聲音 | 同列、同側的相鄰孔導通 |
| `a10`與`e10` | 有聲音 | 左側同一列的五孔組從`a`到`e`導通 |
| `a10`與`f10` | 沒聲音 | 中央溝槽兩側不相通 |
| `a10`與`a11` | 沒聲音 | 不同列不相通 |
| 左側紅色軌最上方與最下方 | 有聲音 | 這片實物的左紅色軌全長導通 |
| 左側紅色軌與左側藍色軌 | 沒聲音 | 左紅、左藍為兩條獨立路徑 |
| 左側紅色軌與右側紅色軌 | 沒聲音 | 左右兩條紅色軌獨立；同色不代表已連接 |

蜂鳴表示電表偵測到低於內部門檻的低阻抗路徑，不表示電阻一定恰好是`0.0Ω`。
沒聲音表示目前測點之間沒有低阻抗連接；先檢查公針、孔位及表筆接觸後，才能
把結果判為麵包板內部獨立。這些結果只適用於本次實物；其他麵包板仍要實測。

#### 紅色`+`、藍色`−`與GND

邊緣紅藍長列稱為**電源軌（power rail）**。紅線、藍線、`+`與`−`都只是印在
塑膠上的接線慣例，不是電源：沒有接USB、電池或電源供應器時，兩條軌都不會
自行產生電壓。

| 實際接法 | 紅色軌的電氣狀態 | 藍色軌的電氣狀態 |
|---|---|---|
| 兩條都沒有接電源 | 未供電 | 未供電 |
| 紅色接ESP32 `3V3`、藍色接`GND` | 相對GND為約+3.3V | 0V參考點／GND |
| 紅色接`5Vin`、藍色接`GND` | 相對GND為約+5V | 0V參考點／GND |
| 顏色接反 | 可能實際成為GND | 可能實際成為正電源；容易誤接，不採用 |

**電壓（voltage）**是兩點之間的電位差。**GND（ground）**是電路選定的0V
共同參考點，不是「裝滿負電」的地方。單一正電源系統通常把電源負端接到GND，
所以藍色`−`軌常被用作GND；這是接線選擇，不是藍色印刷自動造成。若使用正負
雙電源，`−5V`可能低於GND，此時負電源與GND就不是同一個節點。

可以用樓層理解相對關係：GND像定義為0樓的地面，`3V3`像高出3.3V的位置，
`−5V`則像低於地面的地下樓層。選定地面後才能描述其他位置比它高或低。

本週這一段只做斷電通斷測試，不把紅藍軌接到`3V3`、`5Vin`或GND。稍後的GPIO
核心線路直接建立TPO與TPG測點，不依賴電源軌；這可以減少初學階段誤接電源的
機會。

**TP** 是 Test Point（測試點）的縮寫。`TPO`和`TPG`不是麵包板原廠名稱，
而是本實驗定義的標籤：TPO接profile指定的測試輸出，TPG接GND。

先用筆或可移除標籤在板邊寫下：

```text
左外側接線欄：A
右外側接線欄：J
TPO 預留列：________
TPG 預留列：________
```

TPO 與 TPG 必須是兩個不同的空白列，而且不能位於 ESP32 排針占用的列。

### 步驟 3：確認四腳按鈕結構

本課按鈕是**常開瞬時按鈕（normally open momentary pushbutton）**：平常
兩組接點彼此不導通，只有按住時兩組才導通，放開後立即恢復。**導通**表示
兩點之間有可通過測試電流的低阻抗路徑；**開路／不導通**表示路徑中斷。

四支腳不是四個完全獨立的接點。按鈕內部先把兩腳連成A組，另外兩腳連成B組：

```text
A1 ●────● A2       未按：A組與B組分開
        [按鈕]      按住：A組與B組接通
B1 ●────● B2       放開：A組與B組再次分開
```

外觀看見的左、右、上、下不能直接當作固定答案；元件方向改變後，畫面上的位置
也會改變。必須用通斷檔找出哪兩腳原本就是同一組，再選A組的一腳與B組的一腳
作為開關兩端。

本課實物跨在中央溝槽、四腳位於`E27`、`F27`、`E29`、`F29`時，2026-08-29
實測如下：

| 測點 | 未按按鈕 | 判讀 |
|---|---|---|
| `a27`與`j27` | 有聲音 | 第27列兩腳是內部原本相通的同一組 |
| `a27`與`j29` | 沒聲音 | 第27列與第29列是不同組 |
| `a27`與`j29`，按住按鈕 | 有聲音 | 按下後兩組接通 |
| `a27`與`j29`，再次放開 | 沒聲音 | 瞬時按鈕已恢復開路 |

![四腳按鈕跨槽安裝與公對公杜邦線通斷實測](../../docs/images/hardware/actual/tact-switch-6x6mm-4pin-breadboard-continuity-actual.jpg)

第一次用`a27`與`j27`測到蜂鳴不是按鈕損壞，也不是電源短路；兩點只是被按鈕
內部的固定金屬接點連在同一組。把其中一個測點改到第29列後，才能觀察按下與
放開造成的狀態變化。這項結論只適用於已量測的實物與方向；換按鈕或旋轉方向
後要重新確認。

先將按鈕放在桌上觀察並規劃位置：

1. 選兩個相隔約兩列的空白位置，例如第27與第29列。
2. 預定讓四腳進入`E27`、`F27`、`E29`、`F29`，使本體跨在中央溝槽上。
3. 四支腳必須自然對孔，不得扳到明顯變形。
4. 插入後再用通斷檔確認實物的兩組接點，不能只靠外觀猜測。

若實物腳距不同，列號可以改，但必須同時滿足「跨槽」、「四腳自然對孔」與
「通斷量測已找出兩組接點」。

### 步驟 4：用實物確認開發板與麵包板尺寸

**排針（pin header）**是開發板兩側向下的金屬接腳。是否能直接安裝，必須看
排針實際進入哪一欄，不能用開發板黑色PCB邊緣推測。

2026-08-29以`BOARD-T01`和本課400孔麵包板斷電對孔，結果是：

```text
中央實驗區： A  B  C  D  E  │  F  G  H  I  J
                 ▲                         ▲
              左排排針                  右排排針

左排：B3～B24，共22腳
右排：J3～J24，共22腳
```

第3列到第24列包含22個孔位，與開發板左右各22腳、合計44腳相符。這也修正了
先前尚未以實物驗證的`B/I`假設：本批YD板右排實際落在`J`，不是`I`。

![BOARD-T01在400孔麵包板上以B3至B24和J3至J24對孔的實物照片](../../docs/images/hardware/actual/yd-esp32-s3-on-400-breadboard-b3-j24-fit-check.jpg)

這個位置只在左側留下`A`欄，右側`J`欄已被排針占用；右排GPIO沒有同一列的
空孔可接杜邦線。開發板USB端也遮住第27與第29列的外接按鈕。因此本課不把
BOARD-T01直接壓入單片400孔麵包板，也不要求學生購買另一種麵包板；改用已在
必買清單內的公對母杜邦線進行板外連接。

若板子只是懸在孔上，直接垂直拿起。若已壓入：

1. 確認USB、電池與所有其他電源都已拔除。
2. 不拉天線、USB接頭、BOOT或RST按鈕。
3. 兩手捏住PCB兩端的板邊，先把一端抬高約1～2 mm，再換另一端抬高相同距離。
4. 兩端交替、小幅度向上移動，使兩排排針大致平行退出；不可一次把一端大幅
   翹起，也不可用金屬螺絲起子撬板。
5. 取下後檢查44支排針是否仍平行；發現彎針就停止，不自行帶電測試。

### 步驟 5：採用板外公對母杜邦線

**公對母杜邦線（male-to-female jumper wire）**一端是可套在開發板排針上的
母頭，另一端是可插入麵包板的公針。BOARD-T01放在麵包板旁的乾燥、不導電
平面，不讓排針碰到金屬；USB端保持可插拔，線材不能懸吊或拉扯板子。

本週的三條板對板連線都採以下形式：

```text
ESP32板身排針 ← 公對母線的母頭
公對母線的公針 → 麵包板指定五孔組
```

在教師完成GPIO target test並公布profile前，只辨認腳位，不接候選GPIO。公布後
才填寫並逐條連接：

| 訊號 | 板身絲印／profile | 杜邦線顏色 | 麵包板目的地 |
|---|---|---|---|
| 按鈕輸入 | `PIN_BUTTON`：_____ |  | `BTN-A`接點組 |
| 測試輸出 | `PIN_TEST_OUTPUT`：_____ |  | TPO獨立五孔組 |
| 地 | `G`／GND |  | TPG獨立五孔組 |

顏色只用來追蹤線路，不會自動決定正電、負電或GND。母頭必須套在已由絲印與
profile確認的單一排針，不能一次跨到相鄰兩腳；公針垂直插入麵包板，開始彎曲
就停止施力。

### 步驟 6：安裝按鈕

1. 再確認預定的四個孔沒有被 ESP32 或導線占用。
2. 讓按鈕跨過中央溝槽。
3. 四支腳全部對孔後，從按鈕本體正上方平均壓入。
4. 輕推按鈕；它應穩定留在板上，不應只有兩腳勉強插入。
5. 保持完全斷電，用兩條公對公杜邦線延伸測點；找出未按時原本導通的
   第一組並標成`BTN-A`，另一組標成`BTN-B`。
6. 在`BTN-A`與`BTN-B`各選一個測點：未按應不蜂鳴、按住應蜂鳴、放開應再次
   不蜂鳴。三個狀態都符合，才完成按鈕安裝。

### 步驟 7：完成實驗接線

BOARD-T01本次target test先使用下列候選配置。此表記錄目前實物的線色與孔位，不是所有學生都必須使用相同顏色；GPIO4與GPIO5只有在本節所有target test通過後，才能升級為本批板卡的已驗證profile。

| 順序 | 線材與顏色 | ESP32端 | 麵包板端 | 作用 |
|---|---|---|---|---|
| 1 | 棕色公對母 | 右排最上方`GND` | 公頭插`a22` | 建立TPG接地參考列 |
| 2 | 紅色公對母 | 左排候選`GPIO4` | 公頭插`a27` | 連到按鈕`BTN-A`接點組 |
| 3 | 橘色公對母 | 左排候選`GPIO5` | 公頭插`a20` | 建立TPO輸出量測列 |
| 4 | 另一條公對公，顏色另行記錄 | 一端插`b22` | 另一端插`a29` | 從同一TPG分接GND到按鈕`BTN-B`接點組 |

目前按鈕跨中央溝槽，左側兩腳位於`e27`與`e29`。同一列左半部`a`至`e`五孔彼此導通，因此`a27`會連到`e27`，`a29`會連到`e29`。`a20`至`e20`是TPO，`a22`至`e22`是TPG；第20列與第22列彼此不導通。右半部按鈕腳`f27`與`f29`本次不接線。

```text
左半部孔位        目前連接
a20─b20─c20─d20─e20   橘色GPIO5；TPO量測列
a22─b22─c22─d22─e22   棕色GND；b22再分接至a29
a27─b27─c27─d27─e27   紅色GPIO4；e27是按鈕一側
a29─b29─c29─d29─e29   來自TPG；e29是按鈕另一側
```

此接法不使用紅藍電源軌、`3V3`、`5Vin`或右排下方GND。插線期間ESP32必須保持USB斷開。三條公對母線的母頭只套住指定的單一排針，不能同時碰到相鄰排針。

一次只插一條線，每插完一條就在表中打勾：

1. [ ] `PIN_BUTTON`排針 → 公對母線 → 已量測確認的`BTN-A`接點組。
2. [ ] GND排針 → 公對母線 → 預留的TPG空白五孔組。
3. [ ] TPG同一五孔組的另一孔 → 公對公線 → 已量測確認的`BTN-B`接點組。
4. [ ] `PIN_TEST_OUTPUT`排針 → 公對母線 → 預留的TPO空白五孔組。

此接法只使用板上一個 GND：TPG 是共同接地列，再從 TPG 分接到按鈕。四條
杜邦線分別是`PIN_BUTTON→BTN-A`、`GND→TPG`、`TPG→BTN-B`、
`PIN_TEST_OUTPUT→TPO`。

本週的電氣關係必須是：

![Week 2 profile按鈕與測試輸出量測點接線圖](../../docs/images/wiring/week2_gpio4_gpio5.svg)

```text
ESP32 PIN_BUTTON ---- 按鈕的一側
ESP32 GND  --------- 按鈕的另一側
```

另外建立兩個安全量測點，不要直接用表筆在相鄰排針間探測：

```text
ESP32 PIN_TEST_OUTPUT -- 麵包板空白列（標記為 TPO）
ESP32 GND  --------- 麵包板另一空白列（標記為 TPG）
```

後面量電壓時，黑表筆接TPG，紅表筆接TPO，可降低表筆滑動造成短路的
風險。TPO只接`PIN_TEST_OUTPUT`與紅表筆，不接其他模組。

本接法在程式中使用 `INPUT_PULLUP`，不需要額外外接上拉電阻：

- 未按下：讀到 `HIGH`。
- 按下：`PIN_BUTTON`被接到GND，讀到`LOW`。

### 步驟 8：執行上電前接線檢查

不能只看接線「像不像圖片」，必須從訊號起點沿線檢查到終點：

1. 依profile指著`PIN_BUTTON`的板身絲印，確認母頭只套住該排針，再沿線
   走到按鈕`BTN-A`接點組。
2. 指著板身`G`／GND，確認母頭只套住GND排針，再沿線走到TPG，接著從TPG
   走到按鈕`BTN-B`接點組。
3. 依profile指著`PIN_TEST_OUTPUT`的板身絲印，確認母頭只套住該排針，再沿線
   走到標記TPO的獨立列。
4. 再從TPG反向沿線回到板身GND，確認沒有誤套相鄰排針。
5. 確認TPO與TPG不在同一個五孔導通組。
6. 確認沒有任何線接到 `5V`、`3V3` 或未使用的 GPIO。
7. 從正上方拍一張能看清板身絲印與線路終點的照片。

BOARD-T01於2026-08-29完成兩項上電前通斷檢查。第一項以黑表筆接TPG的`c22`、紅表筆接GPIO4按鈕列的`b27`，按鈕放開時不蜂鳴、按住時蜂鳴、放開後恢復不蜂鳴。第二項以黑表筆接`c22`、紅表筆接GPIO5測試列的`b20`，沒有蜂鳴，表示TPO與TPG之間沒有量到低阻抗短接。這些結果證明目前`b22 → a29`分接線、按鈕兩側與GPIO5測試列具備預期的斷電通斷關係，但尚未證明GPIO4、GPIO5、`INPUT_PULLUP`或上電程式通過。完成後把電表切到`OFF`並移開表筆，再由板背`COM` USB-C接頭第一次帶線上電；`PWR`燈亮，五秒內沒有觀察到焦味、煙或異常聲音，之後可見`TX`指示燈週期性閃爍。Arduino IDE的Port選單當時只顯示COM名稱，沒有完整顯示`USB-Enhanced-SERIAL CH343`；選取該COM Port並把Serial Monitor設為`115200 baud`後，實際連續讀到：

```text
uptime_ms=292018
uptime_ms=293018
uptime_ms=294018
uptime_ms=295018
```

相鄰數值約增加`1000 ms`，與舊board-check韌體每秒呼叫一次`Serial.printf()`相符，也反向確認當時選到的是這片ESP32的Port。`TX`是Transmit（傳送）活動指示；此處每送出一行便可能閃一下。Arduino IDE即使關閉，已寫入Flash的韌體仍會在ESP32獲得USB電源時自行執行，因此看見`TX`閃爍不代表短路，也不是GPIO5正在切換。這項結果確認本次帶線上電時UART Serial基準仍正常，但當時執行的仍是舊韌體，不能因此宣稱GPIO4或GPIO5通過。

#### 為什麼第二項通斷測試可以偵測低阻抗短路？

萬用電表的通斷測試模式會使用電表內部電池，在兩支表筆之間施加很小的測試訊號，再判斷測得的路徑是否足夠低阻。不同電表的蜂鳴門檻可能不同，因此不能把『蜂鳴』解讀成電阻一定正好為0Ω；它代表目前量到低於該電表門檻的低阻抗通路。

本次黑表筆放在`c22`。`c22`與棕色GND線所在的`a22`屬於同一五孔組，因此黑表筆位於TPG／GND端。紅表筆放在`b20`；`b20`與橘色GPIO5線所在的`a20`屬於同一五孔組，因此紅表筆位於TPO／候選GPIO5端。量測等效關係是：

```text
紅表筆b20 ── TPO ── 橘色線 ── GPIO5
                                      ?
黑表筆c22 ── TPG ── 棕色線 ── GND
```

若接線錯誤使GPIO5與GND之間出現很低阻抗的直接通路，電表的小測試電流可以完成路徑並持續蜂鳴。此時若上電後再把GPIO5設定為HIGH，GPIO5會嘗試輸出接近3.3V，卻同時被低阻抗路徑拉到0V，可能造成過大電流。因此先在斷電狀態檢查，可在使用系統電源前找出這類錯接。

沒有蜂鳴只表示目前沒有量到低於蜂鳴門檻的GPIO5對GND通路。它不能證明GPIO5腳位選對、HIGH／LOW輸出正常、實際電壓正確，也不能排除高阻漏電、接觸不良或GPIO5與其他接腳之間的錯接。這些項目仍須以接線追蹤、上電程式、Serial輸出與直流電壓量測分別驗證。

### 上電前接線表

| 元件 | 元件腳位 | ESP32 腳位 | 方向／用途 |
|---|---|---|---|
| 按鈕 | `BTN-A`接點組 | `PIN_BUTTON`（填實際GPIO：_____） | 數位輸入 |
| 按鈕 | `BTN-B`接點組 | GND | 按下時接地 |
| TPO 測試列 | 空白麵包板列 | `PIN_TEST_OUTPUT`（填實際GPIO：_____） | HIGH／LOW 電壓測試 |
| TPG 參考列 | 另一空白列 | GND | 黑表筆參考點 |

身邊有同學時，請同學依照上述順序和你一起檢查。獨自操作時，先依第 1 至
第 5 項檢查一次，再從第 5 項反向檢查回第 1 項。確認兩次結果一致，而且沒有
任何線接到 5V，才能插回 USB。

### 本節檢核

- [ ] ESP32未直接插在單片400孔麵包板；排針筆直，板外位置穩定。
- [ ] 三條板對板連線使用公對母杜邦線，母頭各自只套住一支已確認排針。
- [ ] 按鈕四腳自然插入並跨過中央溝槽。
- [ ] 已用通斷檔找出`BTN-A`與`BTN-B`，並完成未按、按住、再次放開三態測試。
- [ ] `PIN_BUTTON`只經按鈕連到GND。
- [ ] `PIN_TEST_OUTPUT`只連到TPO。
- [ ] TPG連到GND，而且TPO、TPG不互通。
- [ ] 接線已依上述方式逐線確認；獨自操作時已完成正向與反向兩次檢查。


## 八、Profile按鈕輸入與測試輸出

### GPIO 與按鈕原理

- **輸入**是 GPIO 接收外部狀態；**輸出**是程式讓 GPIO 產生狀態。
- **HIGH／LOW** 是數位邏輯的兩種狀態。ESP32 的 HIGH 通常接近 3.3V、LOW
  接近 0V，但實際數值仍要用電表量測。
- **`INPUT_PULLUP`** 把 GPIO 設成輸入並啟用內部上拉電阻：按鈕未按時預設
  HIGH，按下接到 GND 時變成 LOW，因此本接法不需外加上拉電阻。
- **浮動**是輸入沒有被明確維持在 HIGH 或 LOW，可能把電氣雜訊誤認成按鈕
  動作；內部上拉可避免未按時浮動。
- **去抖（debounce／debouncing）**是正式技術術語，不是本課自創名稱。Arduino
  官方的[Debounce按鈕範例](https://docs.arduino.cc/built-in-examples/digital/Debounce/)會在
  輸入讀值改變時重新計時，只有讀值維持不變超過指定延遲後才更新按鈕狀態，
  目的是忽略短暫的noise（雜訊或非預期變化）。完整判斷可直接核對
  [Arduino官方`Debounce.ino`原始碼](https://github.com/arduino/arduino-examples/blob/main/examples/02.Digital/Debounce/Debounce.ino)。
- Arduino官方教材把按鈕在一次操作中產生多個短暫開／關脈衝的現象稱為
  **bouncing**；本課使用中文**接點彈跳**及英文`contact bounce／switch bounce`
  說明這個現象。`raw`與`stable`則是本課為了區分「立即讀值」與「已接受狀態」
  所使用的教學標籤，不是Arduino官方範例規定的欄位名稱。
- **去抖實驗**是本課比較不同去抖時間的活動名稱，不是一套全球統一的標準測試
  程序。本課會記錄板卡、按鈕、程式、操作次數與限制，讓結果可以重現。

### 步驟 1：上電前確認與 USB 復電

1. 確認階段 4 的接線檢查項目已全部通過。
2. 確認沒有人握著按鈕、杜邦線或萬用電表表筆。
3. 把 USB 接回原本測試成功的 USB-to-UART 接頭。
4. 觀察數秒；若出現發熱、異味或異常聲音，立刻拔除 USB。
5. 正常時只開 Arduino IDE，不要在上電後移動任何接線。

### 步驟 2：建立按鈕測試程式

`PIN_TEST_OUTPUT`本週不接LED、蜂鳴器、馬達或其他負載；本實驗先用Serial確認
程式已發出HIGH／LOW命令，實際輸出電壓移到Week 3以萬用電表驗證。**負載**是從電路取得能量的裝置，例如 LED、蜂鳴器
或馬達；GPIO 不適合直接供應高電流負載。

#### 程式結構

- `const int`建立不應改變的整數名稱，本程式用它替profile的兩個GPIO命名。
- `bool` 只保存 `true`／`false`，用來表示是否按下。


#### 先讀懂GPIO名稱與數字的對應

正式學生教材在profile完成驗證前使用安全placeholder：

```cpp
const int PIN_BUTTON = -1;
const int PIN_TEST_OUTPUT = -1;
```

`-1`在這裡是「尚未設定」的sentinel value（哨兵值），不是實際GPIO編號；程式的
`profileReady()`會因此阻止硬體功能啟動。教師目前另開候選測試檔，把
`PIN_BUTTON`暫時對應GPIO4、把`PIN_TEST_OUTPUT`暫時對應GPIO5進行BOARD-T01
target test。下列語法拆解使用候選按鈕設定中的GPIO4說明，但在所有實機檢查完成前，
主教材仍保留`-1`，不提前公布為學生固定答案。

候選檔的第一行可逐段閱讀：

| 程式部分 | 意思 | 本次對應 |
|---|---|---|
| `const` | 這個名稱建立後不應在程式執行途中改成別的數值 | 按鈕測試期間固定使用同一支GPIO |
| `int` | 儲存整數的資料型別 | GPIO編號以整數表示 |
| `PIN_BUTTON` | 程式自訂、可讀的名稱 | 看到名稱便知道它負責按鈕輸入 |
| `=` | 在這行把右側數值指定給左側名稱 | 建立名稱與GPIO編號的對應 |
| 候選值`4` | ESP32-S3的GPIO4編號 | 教師target test暫時對應板身絲印`4`；它不是4V，也不是從板邊數來的第4支腳 |
| `;` | 一個C++敘述的結尾 | 少了通常會造成Compile失敗 |

第二行的語法相同，但把`PIN_TEST_OUTPUT`對應到GPIO5：

```text
程式中的 PIN_BUTTON      → 整數4 → 板身GPIO4 → 紅線 → 按鈕
程式中的 PIN_TEST_OUTPUT → 整數5 → 板身GPIO5 → 橘線 → TPO
```

使用名稱的原因是讓後面的程式表達用途。例如
`digitalRead(PIN_BUTTON)`比`digitalRead(4)`更容易看出正在讀按鈕，也能在經過
實機驗證後只修改最前面的對應值，避免在整支程式到處尋找裸露的GPIO數字。

這兩行只建立軟體名稱與編號的對應，尚未設定接腳方向，也不會讓GPIO4或GPIO5
立即產生電壓。真正啟用按鈕輸入與測試輸出的是稍後在`setup()`執行的
`pinMode(PIN_BUTTON, INPUT_PULLUP)`與`pinMode(PIN_TEST_OUTPUT, OUTPUT)`。
GPIO4與GPIO5在BOARD-T01所有target test完成前仍是候選值，不可只因程式能Compile
就視為全班正式profile。
- `unsigned long` 可保存非負整數，本程式用來保存 `millis()` 毫秒時間。
- `setup()` 開機後執行一次；`loop()` 之後持續重複。
- `pinMode()` 設定接腳模式，`digitalRead()` 讀取 HIGH／LOW，
  `digitalWrite()` 輸出 HIGH／LOW。
- `Serial.printf()` 把文字與變數組合成一行 log；`%s` 放文字，`%lu` 放
  `unsigned long` 數值，`\n` 表示換行。

1. 在 Arduino IDE 點 **File → Save As**。
2. 新名稱輸入 `week02_button_groupXX`，將 `XX` 改成兩位數組別。
3. 確認視窗標題已變成新名稱，避免覆蓋前一個 Serial 練習。
4. 按 **Ctrl+A** 全選舊程式，再貼上下列完整程式。
5. 把 `CHANGE_ME` 改成組別，例如第 3 組改成 `03`，保留雙引號。
6. 按 **Ctrl+S**。


In [ ]:
// 由教師公布的同批板卡target-test profile填入；未公布時保持-1。
const int PIN_BUTTON = -1;
const int PIN_TEST_OUTPUT = -1;
const char *GROUP_ID = "CHANGE_ME";

bool stablePressed = false;
bool lastRawPressed = false;
unsigned long changedAtMs = 0;
const unsigned long DEBOUNCE_MS = 30;

bool profileReady() {
  return PIN_BUTTON >= 0 && PIN_TEST_OUTPUT >= 0 &&
         PIN_BUTTON != PIN_TEST_OUTPUT;
}

void setup() {
  Serial.begin(115200);
  delay(500);

  if (!profileReady()) {
    Serial.println("week=2 status=blocked reason=gpio_profile_missing");
    return;
  }

  pinMode(PIN_BUTTON, INPUT_PULLUP);
  pinMode(PIN_TEST_OUTPUT, OUTPUT);
  digitalWrite(PIN_TEST_OUTPUT, LOW);

  Serial.printf("boot: week02 group=%s button-test version=1\n", GROUP_ID);
  Serial.println("state: released input=HIGH test_output=LOW");
}

void loop() {
  if (!profileReady()) return;
  bool rawPressed = digitalRead(PIN_BUTTON) == LOW;
  unsigned long now = millis();

  if (rawPressed != lastRawPressed) {
    lastRawPressed = rawPressed;
    changedAtMs = now;
  }

  if (now - changedAtMs >= DEBOUNCE_MS && rawPressed != stablePressed) {
    stablePressed = rawPressed;
    digitalWrite(PIN_TEST_OUTPUT, stablePressed ? HIGH : LOW);

    Serial.printf(
      "group=%s event=button_changed pressed=%s input=%s test_output=%s time_ms=%lu\n",
      GROUP_ID,
      stablePressed ? "true" : "false",
      stablePressed ? "LOW" : "HIGH",
      stablePressed ? "HIGH" : "LOW",
      now
    );
  }
}


貼上後先做人工檢查：

- [ ] `PIN_BUTTON`與`PIN_TEST_OUTPUT`已填入教師公布profile，不再是`-1`。
- [ ] 兩個值不同，且可由本批板卡target-test紀錄追溯。
- [ ] `GROUP_ID` 已改成自己的組別。
- [ ] 只有一組 `setup()` 與一組 `loop()`。
- [ ] 程式最後的左右大括號數量沒有因複製而缺少。

### 步驟 3：編譯、上傳與開啟 Serial Monitor

#### 先把Verify與Upload分開判讀

2026-08-29在BOARD-T01候選測試中，Arduino IDE的Verify實際顯示：

```text
Sketch uses 302598 bytes (9%) of program storage space.
Maximum is 3145728 bytes.
Global variables use 22136 bytes (6%) of dynamic memory,
leaving 305544 bytes for local variables.
Maximum is 327680 bytes.
```

`302598 bytes`是這次候選程式編譯後的程式空間用量；`3145728 bytes`是目前
Partition Scheme允許單一應用程式使用的上限。`22136 bytes`是編譯時可計算的
全域與靜態變數用量，`305544 bytes`是該記憶體區域在這份摘要中顯示的剩餘量。
兩個百分比都低於上限，因此這次Compile通過容量檢查。

這些數字受到程式版本、Arduino-ESP32版本與Tools設定影響，不是所有學生必須得到的
固定答案。Verify成功只證明編譯工具產生了韌體；它沒有把新韌體寫入ESP32，也沒有
執行按鈕、GPIO4、GPIO5或電壓測試。Verify期間若舊程式仍讓`TX`閃爍，是因為板內
Flash仍保存並執行舊韌體，並非候選程式已經開始運作。

同一次候選測試的Upload實際顯示：

```text
Writing at 0x00059ea0 ... 100.0% 174314/174314 bytes
Wrote 302752 bytes (174314 compressed) at 0x00010000 in 15.5 seconds.
Verifying written data...
Hash of data verified.
Hard resetting via RTS pin...
```

`Writing at`表示工具正把韌體寫入Flash位址，`100.0%`表示本次傳輸完成；
`174314 compressed`是傳輸時壓縮後的資料量，不能當成程式實際占用空間。
`Hash of data verified`表示寫入後資料通過本次完整性比對，`Hard resetting via RTS`
表示CH343自動觸發Reset，讓新韌體開始執行。Verify顯示的302598 bytes與Upload寫入的
302752 bytes不必完全相同，因為兩段摘要計算的映像內容、標頭、對齊或傳輸表示不同。
這些訊息仍未證明GPIO4／GPIO5的實體功能；下一層必須讀Serial事件並實際操作按鈕。

1. 先按 **Verify**。
2. Verify 成功後，再檢查一次 **Tools → Board** 與 **Tools → Port**。
3. 關閉 Serial Monitor。
4. 按 **Upload**，等待完整寫入與重設完成。
5. 開啟 Serial Monitor，設定 `115200`。
6. 按一下 RESET。
7. 第一段文字必須包含自己的組別與 `button-test version=1`。

Upload及自動Reset後，`PWR`應持續亮，表示板子仍獲得電源。`TX`與`RX`是序列
傳送／接收的活動指示，不是「程式正常／異常」狀態燈。舊board-check每秒傳送
`uptime_ms`，因此`TX`會週期性閃爍；按鈕候選程式只在開機及按鈕狀態改變時傳送，
所以閒置時`TX`、`RX`都不亮是合理現象。開機文字可能在Serial Monitor開啟前已經
送完；此時短按一次板上`RST`可重新執行`setup()`並重送文字。不要按`BOOT`，也先
不要按外接按鈕。指示燈現象只能作輔助觀察，仍須以Upload Output與Serial文字確認。

BOARD-T01候選程式在RST後實際輸出：

```text
=== Week 2 GPIO4/GPIO5 candidate test ===
pin_button=4 mode=INPUT_PULLUP
pin_test_output=5 startup=LOW
status=ready expected_released_input=HIGH output=LOW
```

第一行是程式自訂標題，可區分新候選程式與舊board-check；它不是ESP32自動產生的
診斷。第二行表示程式已要求把GPIO4設成啟用內部上拉的輸入；這是軟體設定證據，
實際按鈕路徑仍要操作驗證。第三行表示程式把GPIO5指定為測試輸出並在啟動時先寫
LOW，降低Reset後意外輸出HIGH的風險；真正電壓仍須以TPO對TPG量測。第四行表示
程式已執行到`setup()`末端，而且在按鈕放開的設計狀態下預期輸入HIGH、輸出LOW。
`expected`明確表示這是預期關係，不是這一行同時完成了電壓量測。

如果 Upload 成功但仍看到前一支程式每秒輸出的 `uptime_ms`，表示目前顯示的
可能是錯誤 Port、舊程式或 Upload 未真正完成。先核對 Port 和 Upload Output，
不要改硬體接線。

### 步驟 4：執行按鈕功能測試

按下與放開時，Serial 應出現：

```text
group=03 event=button_changed pressed=true input=LOW test_output=HIGH time_ms=...
group=03 event=button_changed pressed=false input=HIGH test_output=LOW time_ms=...
```

BOARD-T01候選程式第一次實際按下與放開得到：

```text
event=button_changed pressed=true input=LOW output=HIGH time_ms=104914
event=button_changed pressed=false input=HIGH output=LOW time_ms=105096
```

`pressed=true input=LOW`表示程式在去抖後確認按鈕已把GPIO4接到GND；
`pressed=false input=HIGH`表示放開後GPIO4由內部上拉恢復HIGH。兩筆事件只有一次
按下與一次放開，未觀察到同一次操作被重複計數。`time_ms`是本次Reset後經過的
毫秒數，不是時鐘時間；`105096 − 104914 = 182 ms`，表示兩個穩定事件之間約
182 ms。去抖流程要等待`DEBOUNCE_MS`，所以這個差值適合描述程式確認到的狀態間隔，
不當作按鈕機械接點的精密量測。

同一行的`output=HIGH／LOW`是程式執行`digitalWrite()`後回報的命令狀態，只證明
程式走到對應分支。它不能單獨證明GPIO5排針真的產生約3.3V／0V；實體輸出仍須以
TPO對TPG的直流電壓量測確認。

請照固定節奏操作：

1. 先放開按鈕，確認沒有持續重複事件。
2. 按下並保持約一秒，只應新增一筆 `pressed=true`。
3. 放開並等待約一秒，只應新增一筆 `pressed=false`。
4. 再慢速做兩次。
5. 若一次動作印出很多筆，不要急著增大 `DEBOUNCE_MS`；先排除接觸不良和
   按鈕插錯方向。

如果按下沒有反應：

1. 拔除 USB。
2. 檢查是否真的接到profile指定的`PIN_BUTTON`與GND。
3. 確認按鈕方向及是否跨過麵包板中央溝槽。
4. 用通斷檔確認按鈕按下時兩側導通。
5. 接回 USB，按 RESET，再看 Serial。

所有線路修正均應先拔除 USB，不得在通電狀態下移動接線。

### 步驟 5：依現象進行故障排除

| Serial 現象 | 最可能方向 | 下一個動作 |
|---|---|---|
| 一上電就顯示 `pressed=true` | `PIN_BUTTON`持續接地 | 拔USB，檢查按鈕方向及`PIN_BUTTON`、GND是否在同一導通組 |
| 按下、放開都沒有事件 | `PIN_BUTTON`未經按鈕接到GND | 拔USB，逐線摸查，再做通斷測試 |
| 一次按壓出現很多事件 | 接點彈跳或接觸不良 | 確認按鈕完全插入，再比較去抖設定 |
| 事件正常但組別錯誤 | 程式未改或舊程式 | 修改 `GROUP_ID`、Save、Upload、RESET |
| 完全沒有 Serial 文字 | Port／baud／USB 問題 | 回到階段 3 的 Serial 排錯，不動硬體線 |

排錯前先保存畫面。凡是要碰線，一律先拔 USB。修正後重新逐線檢查；獨自
操作時，必須再做一次正向與反向檢查。

### 步驟 6：執行五次重複性測試

每次完整按下再放開，記錄：

| 次數 | 按下顯示 `true/HIGH` | 放開顯示 `false/LOW` | 備註 |
|---:|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |
| 4 |  |  |  |
| 5 |  |  |  |

五次中任何一次失敗，都先留下 log，再修正並重新計算五次。

BOARD-T01實際完成五次後，Serial事件依序為：

| 次數 | 按下事件時間 | 放開事件時間 | 兩個穩定事件相差 | 結果 |
|---:|---:|---:|---:|---|
| 1 | 104914 ms | 105096 ms | 182 ms | LOW／HIGH事件各一筆 |
| 2 | 227156 ms | 228344 ms | 1188 ms | LOW／HIGH事件各一筆 |
| 3 | 229836 ms | 230839 ms | 1003 ms | LOW／HIGH事件各一筆 |
| 4 | 232135 ms | 233159 ms | 1024 ms | LOW／HIGH事件各一筆 |
| 5 | 234757 ms | 235947 ms | 1190 ms | LOW／HIGH事件各一筆 |

五次均嚴格交替為`pressed=true input=LOW`與`pressed=false input=HIGH`，沒有重複、
漏失或卡在單一狀態。第一次是較短的182 ms點按；後四次依操作要求約保持1秒。
這項證據支持BOARD-T01目前接線下GPIO4候選輸入、內部上拉、按鈕路徑與30 ms
去抖流程具備五次重複性。每行的`output=HIGH／LOW`仍是程式命令紀錄，不能取代
GPIO5的TPO對TPG直流電壓量測。


## 九、萬用電表：斷電通斷驗證

依課堂公布的量測站順序使用萬用電表；Week 2只在完全斷電時做麵包板與按鈕通斷。
兩人一組時，由量測者控制表筆，另一人負責按鈕與記錄；兩人的手不要同時伸進
電路。獨自操作時，不勉強同時拿兩支表筆與按鈕，應先用麵包板固定按鈕，再以
杜邦線延伸測點；接觸仍不穩時停止並重新固定，不用手硬撐出一筆數字。

**萬用電表（multimeter）**是一台可選擇不同功能量測電壓、電阻等數值的
儀表；紅、黑兩條尖端量測線稱為**表筆（probe）**。本教材以教師現有的
A830L 為操作示例；若正式材料清單安排功能相當的其他型號，符號與檔位位置
可能不同，操作時仍以實物標示為準。

### 先讀懂A830L：符號、顏色、數字與單位

旋鈕周圍的藍色與黑色只是這個製造商協助區分功能的印刷方式，不是通用標準，
也不表示「藍色安全、黑色危險」。每次選檔都要依序讀三項資訊：

1. **功能符號**：`Ω`、`V⎓`、`V~`或`A⎓`。
2. **量程數字**：例如`200`、`2k`或`20`。
3. **單位與前綴**：`Ω`、`V`、`A`以及`m`、`k`、`M`。

| A830L標示 | 代表的功能 | 本週用途 |
|---|---|---|
| 藍色`Ω`與`200`、`2k`、`20k`等 | 電阻及其量程 | 斷電後測電阻與開關 |
| 藍色聲音符號 | 導通蜂鳴 | 斷電後判斷兩點是否低阻抗相通 |
| 藍色二極體符號 | 二極體測試 | 本週不用 |
| 黑色`V⎓` | 直流電壓 | Week 3量3V3、5Vin與GPIO LOW／HIGH |
| 黑色`V~` | 交流電壓 | 本課不用；不得量市電插座 |
| 黑色`A⎓`與`mA`／`10A` | 直流電流 | 本週不用 |
| `hFE`與`E C B`插孔 | 電晶體測試 | 本週不用 |

`V`是伏特（volt），為電壓單位；`A`是安培（ampere），為電流單位；`Ω`是
歐姆（ohm），為電阻單位。常用前綴如下：

```text
1 A = 1000 mA
1 V = 1000 mV
1 kΩ = 1000 Ω
1 MΩ = 1,000,000 Ω
```

同一個`200`放在不同功能區，意義並不相同。`Ω 200`是約200Ω的電阻量程；
`V⎓ 200`是約200V的直流電壓量程；`A⎓ 200m`則是200mA的直流電流量程。
不能只看到數字就轉動旋鈕。

#### `Ω 200`畫面中的數字如何判讀

使用`Ω 200`時，這台手動量程電表可顯示到約199.9Ω。下列例子都以旋鈕確實
位於`Ω 200`為前提：

| 畫面 | 判讀 | 實物例子 |
|---:|---|---|
| `0.6` | 0.6Ω，低電阻通路 | 兩表筆穩定短接，或按鈕確實閉合 |
| `24.8` | 24.8Ω | 對短接表筆而言偏高，可能接觸不穩 |
| `117.1` | 117.1Ω | 對短接表筆而言明顯偏高 |
| 最左側固定的`1` | 超過本量程或開路，不是1Ω | 表筆分開，或按鈕未按 |

「高電阻不合理」必須連同被測物一起判斷。導線或閉合按鈕原本應提供接近0Ω的
金屬通路，因此量到數十至上百歐姆時應先檢查接觸；未按下的常開按鈕本來就是
開路，顯示最左側的`1`反而正確；220Ω電阻本來就超過`Ω 200`量程，也會顯示
最左側的`1`，此時應換到下一個較高的電阻量程。

#### 用實物理解電壓、電流與電阻

- **電壓（voltage）**是兩點之間推動電荷的電位差，可暫時類比為水管兩端的
  壓力差，符號是`V`。
- **電流（current）**是通路中電荷流動的速率，可暫時類比為每秒流過的水量，
  符號是`I`，單位是`A`。
- **電阻（resistance）**表示通路阻礙電流的程度，可暫時類比為水管對水流的
  阻礙，符號是`R`，單位是`Ω`。

三者的基本關係是歐姆定律：

```text
V = I × R
I = V ÷ R
R = V ÷ I
```

例如3.3V加在330Ω電阻兩端時：

```text
I = 3.3 V ÷ 330 Ω
  = 0.01 A
  = 10 mA
```

如果改成1000Ω，電流約為3.3mA；如果改成100Ω，理想計算約為33mA。其他
條件相同時，電阻增加會使電流減少，電阻減少會使電流增加。

##### 電流方向與完整回路

本課使用**傳統電流方向（conventional current）**：在電源外部電路中，電流由
較高電位經過負載流向較低電位。金屬中的電子移動方向相反，但電路圖、Arduino
教材及一般量測判讀都使用傳統方向。

```text
電源正端／3.3V
       │
       ↓ 傳統電流
  電阻、LED或其他負載
       │
       ↓
GND／電源負端
       │
       └──────── 回到電源內部
```

電流必須有完整閉合回路才會持續流動。GND不是讓電流消失的洞，而是回到電源的
共同路徑與電壓參考。只有3.3V與GND存在但中間路徑斷開時，不會形成持續電流。

##### 電流不是越大越好，也不是越小越好

每種元件、導線與電源都有可承受範圍：

```text
電流太小   → 元件可能無法工作
電流適當   → 產生預期的光、聲音、動作或訊號
電流過大   → 接點、導線、PCB或晶片可能發熱與損壞
```

電流流過設計好的負載時，能量可轉成光、聲音、運動或運算。若用導線直接把
3.3V或5V接到GND，電流會繞過負載，沿非常低電阻的捷徑流動，這就是
**短路（short circuit）**。

```text
正常：3.3V ── 電阻／負載 ── GND
短路：3.3V ─────低電阻導線───── GND
```

用`0.1Ω`只作概念計算：`3.3V ÷ 0.1Ω = 33A`。實際USB與板卡通常無法供應
這個理想電流，會先限流、掉電、重啟、發熱或損壞；這不是可以用實物嘗試的
實驗。

電功率可寫成：

```text
P = V × I
P = I² × R
```

某段接點或導線仍有少量電阻。電流增加時，這些位置產生的熱會快速增加，所以
短路即使發生在低電壓電路，仍可能造成危險。

「電源可提供3A」表示它在規格條件下最多有能力供應到該範圍，不表示它會強迫
3A流進任何設備。實際電流由電壓、負載及控制電路共同決定。不是每個負載都要
額外串聯一顆電阻；馬達線圈、舵機控制器與晶片本身已有不同電氣特性，但仍必須
使用符合規格的電源與驅動方式。ESP32 GPIO是訊號腳，不能拿來直接供應馬達或
舵機所需的大電流。

本週按鈕使用`INPUT_PULLUP`。按鈕放開時，通往GND的路徑斷開，GPIO被內部上拉
電阻維持在`HIGH`；按下時，GPIO經按鈕連到GND而讀到`LOW`。內部上拉電阻會限制
電流，因此這不是用按鈕把3.3V直接短路到GND。

同一個「接近0Ω」結果必須依情境判讀：閉合按鈕或同一麵包板節點接近0Ω是正常；
電源正端與GND之間在沒有合格負載時接近0Ω則代表短路風險。

#### 照片上的`10A`不是`10V`

A830L左下插孔標示的是`10A MAX UNFUSED`。`A`是電流單位，不是電壓；
`UNFUSED`表示這條大電流輸入沒有保險絲保護。量電壓時，電表以高內阻並聯在
兩個測點之間；量電流時，必須先中斷原通路，再把低內阻的電表串聯進電路。
若紅表筆插在`10A`孔，卻像量電壓一樣直接跨接電源與GND，會形成近似短路，
可能使電池、導線、板卡或電表發熱及損壞。

10V本身與10A不是同一件事。一般乾燥完整皮膚接觸10V的觸電風險通常較低，
但10V遠高於ESP32-S3 GPIO使用的3.3V邏輯，直接接入GPIO仍可能立即損壞零件；
能供應大電流的低壓電源在短路時也可能造成發熱。這門課只量USB、ESP32與電池
等低壓直流電路，不量牆壁插座。

三種量測的接法與通電狀態不能互換：

| 要量什麼 | 電路狀態 | 電表接法 |
|---|---|---|
| 電阻／導通 | 必須斷電 | 電表跨接被測物兩端；使用`Ω`或蜂鳴檔 |
| 直流電壓 | 量測時通電 | 電表並聯在兩測點；使用`V⎓` |
| 直流電流 | 量測時通電，但要先中斷原通路 | 電表串聯進電路；本週不做 |

### 本週量測範圍

本週只做斷電通斷，因為它直接支援目前的麵包板與按鈕線路。220Ω、1kΩ、
10kΩ散裝電阻，以及3V3、5Vin與GPIO測試輸出的直流電壓，統一移到Week 3
感測器接線前完成。此處保留電壓、電流、電阻與檔位說明，讓學生先能解釋
符號與安全差異；本週不得把旋鈕切到電壓或電流檔進行額外試量。

### 斷電通斷量測

#### 步驟1：表筆安裝與斷電確認

- 電表的 **`COM`** 是共同參考插孔，接黑表筆；它不是 Windows 的 COM Port。
- **`VΩmA`** 是量電壓、電阻與小電流時使用的紅表筆插孔。本週只量電壓與
  電阻／通斷。
- **`10A`** 是大電流量測插孔，本週完全不用。

1. 從 ESP32 端拔除 USB，確認電源燈熄滅。
2. 萬用電表旋鈕先轉到 `OFF`。
3. 黑表筆插入標示 `COM` 的孔，插到底。
4. 紅表筆插入標示 `VΩmA` 的孔，**不可插在 `10A` 孔**。
5. 輕拉兩條表筆接頭，確認沒有鬆脫。
6. 確認桌上沒有外接電池或其他電源。

#### 步驟2：選擇通斷或200Ω檔

**通斷檔**用來判斷兩點是否低阻抗相通，必須先讓待測電路斷電。`Ω` 是電阻
單位歐姆；`200 Ω` 是這次選擇的量程。畫面顯示 `OL` 或最左側的 `1`，通常
代表開路或超出量程，不是測得 1 Ω。

1. 這支A830L有蜂鳴／聲波符號，可選通斷檔；需要看實際數值時，改選電阻區
   的`200`，表示約200Ω的電阻量程。
2. 先把紅、黑表筆分開：蜂鳴應停止；在`Ω 200`應顯示最左側的`1`。
3. 讓紅、黑表筆的金屬尖端穩定接觸：應發出聲音；在`Ω 200`應降到`0.x`或
   數歐姆。這個數值包含兩條表筆、接觸點及電表內部的少量電阻，不要求恰好
   是`0.0`。
4. 若只是擦到表筆側面或尖端壓力不穩，讀值可能在數十至上百歐姆之間跳動。
   蜂鳴器只依內部門檻判斷是否發聲，有聲音不等於電阻恰好為零。
5. 保持尖端接觸並分別輕動黑線、紅線。固定接觸時若某條線一動就回到`1`或
   大幅跳動，重新插緊該表筆；仍無法穩定時停止使用並回報。
6. 再把兩表筆分開，確認畫面回到最左側的`1`。短接與分開結果都合理，才拿
   這台電表判斷其他零件。

正式課前試量曾出現「碰觸時蜂鳴，但畫面在數十至上百歐姆跳動」的現象。
改用`Ω 200`、確認小數點並讓尖端確實接觸後，讀值降到`0.x Ω`。這個過程說明
排錯不能只聽蜂鳴，也不能把快速變動的`19.8`看成`198`；必須同時記錄檔位、
小數點、單位、接觸位置及穩定後的數字。

#### 步驟3：麵包板與按鈕量測

下列每次量測，表筆各碰一個孔中的金屬接點或同列杜邦線金屬端，不能讓兩支
表筆尖端互相碰到。

1. 先量同一側同一五孔組，例如 `A27` 與 `E27`：應導通。
2. 再量中央溝槽兩側同一列，例如 `E27` 與 `F27`：應不導通。
3. 將兩條公對公杜邦線分別插到預計作為`BTN-A`與`BTN-B`的接點組；表筆
   只碰線材另一端外露的金屬公針，不能把較粗的表筆硬插進麵包板。
4. 兩人一組時由一人固定表筆、另一人操作按鈕。獨自操作時，把兩條線的外露
   公針分開平放，讓線材與表筆靠在桌面，再用空出的手按按鈕。兩支表筆不可
   直接互碰，否則會繞過按鈕而蜂鳴。
5. 不按按鈕時讀值：應顯示最左側的`1`或不蜂鳴，表示兩組開路。
6. 保持測點不變並按住按鈕：應降到`0.x`或數歐姆並導通。若先看到約`5x Ω`，
   重新穩定表筆與按鈕；穩定接觸後若降到`0.x Ω`，先前高值屬於量測接觸，
   不能當作按鈕的真實閉合電阻。
7. 放開按鈕：應再次顯示最左側的`1`或不蜂鳴。

若第5步一開始就導通，不要立刻判定按鈕故障。兩個測點可能位於內部原本相通的
同一組：保留一個測點，把另一個改到對角線上的另一列，再依「未按、按住、放開」
順序重測。問題排除前不得上電。

本課實物第一次把線接在`a27`與`j27`時，未按就蜂鳴，證明第27列兩腳是同一組；
把右側線移到`j29`後，實測得到「未按沒聲音、按住有聲音、放開再度沒聲音」。
這三個結果共同證明按鈕是常開瞬時型，也說明為何不能只把外觀左、右當成兩端。

教師實物驗證接法：

![四腳按鈕跨槽安裝與通斷實測](../../docs/images/hardware/actual/tact-switch-6x6mm-4pin-breadboard-continuity-actual.jpg)

| 2026-08-29實際測點與狀態 | 蜂鳴結果 | 判定 |
|---|---|---|
| `a27`與`j27`，未按 | 有聲音 | 同一個固定接點組，不是有效的開關兩端 |
| `a27`與`j29`，未按 | 沒聲音 | 兩組彼此開路 |
| `a27`與`j29`，按住 | 有聲音 | 按下使兩組導通 |
| `a27`與`j29`，放開 | 沒聲音 | 放開後恢復開路 |

學生記錄結果：

| 狀態 | 蜂鳴／顯示 | 判定 |
|---|---|---|
| 未按下 |  | 不導通 |
| 按住 |  | 導通 |
| 再次放開 |  | 恢復不導通 |

### 本節檢核

- [ ] 所有通斷量測都在USB與外部電源拔除後進行。
- [ ] 同一五孔組導通，中央溝槽兩側與不同列不導通。
- [ ] 按鈕未按不導通、按住導通、放開恢復不導通。
- [ ] 能說明蜂鳴代表低阻抗路徑，不代表讀值一定恰好為0Ω。
- [ ] 量測後萬用電表已轉回`OFF`，表筆已移離線路。

Week 3會在相同安全原則下加入散裝電阻與上電直流電壓；完成Week 2時不要求
填寫尚未執行的電阻或電壓數字。


## 十、Week 2 Discussion：按鈕去抖

本節只討論一個核心問題：人只按一次按鈕，ESP32為什麼可能讀到多次變化，
程式又如何把這些立即讀值整理成一次有效事件。先理解什麼是去抖，再說明
按鈕彈跳如何產生及程式如何完成去抖，最後才從同一概念延伸三項學生練習。

### 10.1 什麼是按鈕去抖

本節以Arduino官方[Debounce按鈕範例](https://docs.arduino.cc/built-in-examples/digital/Debounce/)
及其[`Debounce.ino`原始碼](https://github.com/arduino/arduino-examples/blob/main/examples/02.Digital/Debounce/Debounce.ino)
為基準。**按鈕去抖（button debouncing）**是程式不把每一次短暫HIGH／LOW變化
都立即當成有效事件，而是先等待讀值穩定，再接受新的按鈕狀態。Arduino官方
範例說明這樣做是為了忽略noise（短暫雜訊或非預期變化）。

去抖不是讓機械接點停止彈跳，也不是讓按鈕本身變慢。它改變的是程式的判斷：
立即讀值可以先被觀察，但只有符合穩定條件的變化才成為系統正式使用的事件。

**raw讀值**是本課對`digitalRead()`當下讀到之HIGH或LOW的教學名稱；
**stable狀態**是本課對維持指定時間後、程式正式接受之狀態的教學名稱。Arduino
官方原始碼使用`reading`、`buttonState`與`lastButtonState`等名稱，沒有規定所有
程式都必須使用`raw`或`stable`。本課使用這兩個標籤，是為了讓log能清楚區分
「現在讀到什麼」和「系統是否已接受」。

### 10.2 按鈕彈跳怎樣產生

本課的四腳輕觸按鈕內部有會移動的金屬接點。若只畫出理想結果，按下一次應為：

```text
按下前：HIGH
按下後：LOW
```

但立即讀值可能在正式穩定前短暫改變數次。下圖是用來理解官方演算法的
**教學示意**，不是本片按鈕已量到的波形：

```text
人的動作：     放開 -------------------------- 按下
理想輸入：     HIGH --------------------------- LOW
可能的raw讀值：HIGH -- LOW -- HIGH -- LOW ------ LOW
```

人的動作仍是「按一次」，ESP32的`loop()`卻會快速重複讀取GPIO。如果程式把每次
HIGH／LOW變化都立即當成新狀態，一次動作便可能產生多個事件。機械接點造成的
這類快速變化通常稱為**接點彈跳（contact bounce／switch bounce）**。實際是否
出現、持續多久，必須以自己的原始log或量測證據判斷，不能由示意圖直接推定。

自然產生的是按鈕的**彈跳**；**去抖**則是硬體或程式對這些短暫變化所做的處理。
因此「怎樣產生按鈕去抖」在本節更精確的問題是：程式怎樣根據raw讀值產生一個
可以正式使用的stable狀態。

### 10.3 程式怎樣完成去抖

Arduino官方程式不在讀到一次變化時立刻更新按鈕狀態；它先記錄讀值改變的時間，
等讀值維持不變超過`debounceDelay`後才接受新狀態。本週程式沿用這個核心判斷，
並改寫成可同時輸出raw與stable證據：

1. raw讀值一改變，就記下改變時間`changedAtMs`。
2. 若raw又改變，重新記時；先前等待不算完成。
3. 只有raw連續不變至少`DEBOUNCE_MS`，而且與舊stable狀態不同，才接受新狀態。
4. 接受後才更新`stablePressed`、送出GPIO5命令及寫入stable log。

`DEBOUNCE_MS = 30`表示必須穩定30毫秒，即0.03秒。ESP32在這30毫秒內沒有停止；
它仍持續執行`loop()`並檢查raw讀值。這種使用`millis()`的寫法稱為**non-blocking**
（非阻塞）等待，因為沒有用`delay(30)`讓整個程式暫停。

#### 完整示範：為什麼計時會重新開始

假設`DEBOUNCE_MS = 30`，按下瞬間得到下列raw讀值：

| 時間 | raw讀值 | 程式動作 |
|---:|---|---|
| 1000 ms | LOW | raw改變，從1000 ms重新計時 |
| 1003 ms | HIGH | raw又改變，先前3 ms作廢，從1003 ms重新計時 |
| 1008 ms | LOW | raw再次改變，從1008 ms重新計時 |
| 1038 ms | 仍為LOW | LOW已連續穩定30 ms，正式接受`pressed=true` |

因此這次人只按一下；raw改變三次，但stable只接受一次。若程式在1000 ms第一次
看到LOW就立即接受，1003 ms可能又被當成放開，1008 ms又被當成第二次按下。

### 10.4 Discussion練習一：自己追蹤去抖計時

不要執行程式，先在紙上分析下列情況。`DEBOUNCE_MS = 30`，原本stable是HIGH：

| raw改變時間 | 新raw讀值 | 此刻是否接受新stable狀態？ | 若不接受，新的計時起點 |
|---:|---|---|---:|
| 5000 ms | LOW |  |  |
| 5006 ms | HIGH |  |  |
| 5011 ms | LOW |  |  |
| 5018 ms | HIGH |  |  |
| 5020 ms | LOW |  |  |

回答三個問題：

1. raw最後一次改變後，最早到哪一個`time_ms`才可以接受LOW？
2. 人的動作是一個按下，raw總共改變幾次？
3. stable最終應接受幾次按下？

完成條件：答案同時使用「raw」、「stable」與「連續穩定30 ms」說明，不只填一個
時間數字。

### 10.5 診斷程式：同時觀察raw與stable

下一支程式會印出兩種事件：

- `event=raw_changed`：GPIO立即讀值改變，尚未經去抖。
- `event=stable_changed`：raw已連續穩定到指定時間，程式正式接受事件。

程式中的Serial輸出會增加每次loop所需時間，因此它適合課堂觀察與比較，不是量測
機械彈跳波形的精密儀器。若要觀察微秒級電氣波形，需要示波器或邏輯分析儀；本週
不要求購買或使用。

1. 點 **File → Save As**，另存為`week02_debounce_groupXX`。
2. 以 **Ctrl+A** 全選舊程式，貼上下一格完整程式。
3. 把`CHANGE_ME`換成組別，並填入教師公布的GPIO profile；未公布時保持`-1`，
   程式會阻止實驗啟動。
4. 第一輪保留`DEBOUNCE_MS = 30`，不要先改成其他數字。

Repository中的可執行原始檔是
[`week02_button_debounce_lab.ino`](../../examples/week02_button_debounce_lab/week02_button_debounce_lab.ino)；
它與下一格notebook程式必須保持一致。


In [ ]:
// 由教師公布的同批板卡target-test profile填入；未公布時保持-1。
const int PIN_BUTTON = -1;
const int PIN_TEST_OUTPUT = -1;
const char *GROUP_ID = "CHANGE_ME";

const unsigned long DEBOUNCE_MS = 30;

bool experimentReady = false;
bool lastRawPressed = false;
bool stablePressed = false;
unsigned long changedAtMs = 0;
unsigned long rawEdgeCount = 0;
unsigned long acceptedPressCount = 0;
unsigned long acceptedReleaseCount = 0;

bool profileReady() {
  return PIN_BUTTON >= 0 && PIN_TEST_OUTPUT >= 0 &&
         PIN_BUTTON != PIN_TEST_OUTPUT;
}

void setup() {
  Serial.begin(115200);
  delay(500);

  if (!profileReady()) {
    Serial.println("week=2 status=blocked reason=gpio_profile_missing");
    return;
  }

  // 先把輸出暫存值設為LOW，再啟用OUTPUT，降低啟動時意外HIGH的風險。
  digitalWrite(PIN_TEST_OUTPUT, LOW);
  pinMode(PIN_TEST_OUTPUT, OUTPUT);
  pinMode(PIN_BUTTON, INPUT_PULLUP);

  lastRawPressed = digitalRead(PIN_BUTTON) == LOW;
  stablePressed = false;
  changedAtMs = millis();

  if (lastRawPressed) {
    Serial.println("week=2 status=blocked reason=release_button_then_reset");
    return;
  }

  experimentReady = true;
  Serial.printf(
    "boot: week02 group=%s debounce-diagnostic debounce_ms=%lu\n",
    GROUP_ID,
    DEBOUNCE_MS
  );
  Serial.println("state: raw_pressed=false stable_pressed=false test_output=LOW");
}

void loop() {
  if (!experimentReady) return;

  const bool rawPressed = digitalRead(PIN_BUTTON) == LOW;
  const unsigned long now = millis();

  if (rawPressed != lastRawPressed) {
    lastRawPressed = rawPressed;
    changedAtMs = now;
    rawEdgeCount++;

    Serial.printf(
      "group=%s event=raw_changed raw_pressed=%s raw_edge=%lu time_ms=%lu\n",
      GROUP_ID,
      rawPressed ? "true" : "false",
      rawEdgeCount,
      now
    );
  }

  if (now - changedAtMs >= DEBOUNCE_MS && rawPressed != stablePressed) {
    stablePressed = rawPressed;
    digitalWrite(PIN_TEST_OUTPUT, stablePressed ? HIGH : LOW);

    if (stablePressed) {
      acceptedPressCount++;
    } else {
      acceptedReleaseCount++;
    }

    Serial.printf(
      "group=%s event=stable_changed pressed=%s input=%s "
      "test_output=%s accepted_press=%lu accepted_release=%lu "
      "stable_for_ms=%lu debounce_ms=%lu time_ms=%lu\n",
      GROUP_ID,
      stablePressed ? "true" : "false",
      stablePressed ? "LOW" : "HIGH",
      stablePressed ? "HIGH" : "LOW",
      acceptedPressCount,
      acceptedReleaseCount,
      now - changedAtMs,
      DEBOUNCE_MS,
      now
    );
  }
}


#### 逐段讀懂診斷程式

`lastRawPressed`保存前一次立即讀值，`stablePressed`保存已正式接受的狀態。它們
可能暫時不同：例如raw剛變成LOW，但尚未穩定30 ms時，raw表示已按下，stable仍
維持放開。

```cpp
if (rawPressed != lastRawPressed) {
  lastRawPressed = rawPressed;
  changedAtMs = now;
}
```

這段在raw每次改變時重設計時起點。彈跳造成的LOW→HIGH→LOW會讓`changedAtMs`
一再更新，不會把不連續的時間加在一起。

```cpp
if (now - changedAtMs >= DEBOUNCE_MS && rawPressed != stablePressed)
```

這一行必須同時滿足兩個條件：

1. `now - changedAtMs >= DEBOUNCE_MS`：目前raw已連續穩定足夠久。
2. `rawPressed != stablePressed`：穩定後的結果真的與上次接受狀態不同。

若只有第一個條件，按鈕維持不動時可能在每次loop重複印出事件。第二個條件確保
每次穩定狀態轉換只接受一次。

`rawEdgeCount`只要raw改變就增加；`acceptedPressCount`與
`acceptedReleaseCount`只在去抖後接受狀態時增加。10次完整按下與放開的理想結果
是接受10次press及10次release；raw理想上改變20次，但若有彈跳或接觸擾動，可能
多於20次。

#### Discussion練習二：觀察自己的按鈕

1. 放開外接按鈕，再按 **Verify**。Compile成功只表示程式可建立，尚未觀察按鈕。
2. 關閉Serial Monitor後按 **Upload**，等待hash驗證與Reset完成。
3. 開啟115200-baud Serial Monitor。若出現`release_button_then_reset`，放開按鈕，
   再按板上RST。
4. 核對boot行中的組別及`debounce_ms=30`。
5. 慢慢按下並保持約1秒，再放開約1秒。先只做一次。
6. 在log中用不同記號標出`raw_changed`與`stable_changed`。
7. 看懂單次log後按一次RST，核對計數已重新從0開始；示範的第一次不列入正式資料。
8. 再做5次慢速按壓及5次快速點按，總共10次正式操作。快速點按只要求同一位
   操作者盡量使用相同方式，不假裝每次時間完全相同。

一個沒有觀察到額外raw變化的例子可能是：

```text
event=raw_changed raw_pressed=true raw_edge=1 time_ms=...
event=stable_changed pressed=true ... stable_for_ms=30 debounce_ms=30 ...
event=raw_changed raw_pressed=false raw_edge=2 time_ms=...
event=stable_changed pressed=false ... stable_for_ms=30 debounce_ms=30 ...
```

若同一次按下在stable接受前出現多筆快速交替的`raw_changed`，這與接點彈跳或線路
接觸擾動相符；先保存log，再確認線路沒有鬆動。若只出現理想的兩筆raw變化，應寫
「本次未觀察到額外raw變化」，不能反過來宣稱機械按鈕永遠不會彈跳。

| 紀錄時點 | 累積實際次數 | 此刻`raw_edge` | 此刻`accepted_press` | 此刻`accepted_release` | 觀察到的額外raw變化或漏失 |
|---|---:|---:|---:|---:|---|
| 完成5次慢速操作後 | 5 |  |  |  |  |
| 再完成5次快速操作後 | 10 |  |  |  |  |

完成條件：能從自己的log指出至少一組raw與stable的對應，接受的press及release各為
多少，並使用「觀察到」或「未觀察到」描述結果，不把預期現象寫成實測結果。


### 10.6 Discussion練習三：比較`0／10／30／100 ms`

到這裡才開始改`DEBOUNCE_MS`。四個數字表示raw必須連續穩定多久，程式才接受：

| 設定 | 換算 | 意義 | 可能代價 |
|---:|---:|---|---|
| 0 ms | 0秒 | 不等待，raw變化可立即成為stable事件 | 可能把彈跳接受成多次事件 |
| 10 ms | 0.01秒 | 只忽略短於約10 ms的不穩定 | 較長彈跳仍可能通過 |
| 30 ms | 0.03秒 | 必須穩定約30 ms | 每次接受至少晚約30 ms |
| 100 ms | 0.1秒 | 必須穩定約100 ms | 很短的點按可能在被接受前就放開 |

這些是設定含義與可能性，不是預先指定的實測答案。按鈕、線路、程式執行時間與
操作方式都會影響結果。


#### 3-1：先寫預測

在執行前填寫。**預測**是根據目前理解提出可能結果，不是猜教師答案；實際結果
不同時保留原預測，才能比較自己原先的理解。

| 設定 | 預測10次操作會接受幾次press／release | 預測raw是否有額外變化 | 預測快速點按是否可能漏失 | 理由 |
|---:|---|---|---|---|
| 0 ms |  |  |  |  |
| 10 ms |  |  |  |  |
| 30 ms |  |  |  |  |
| 100 ms |  |  |  |  |


#### 3-2：每輪只改去抖時間

「控制變因」在本題的白話意思是：**每一輪只改`DEBOUNCE_MS`，其他條件盡量
維持一樣**。使用同一片ESP32、同一顆按鈕、同一接線、同一位操作者，以及相同的
「5次慢按＋5次快速點按」順序。若同時換線或換按鈕，就無法知道結果是由哪個改變
造成。

1. 先將`DEBOUNCE_MS`改成0，按 **Ctrl+S → Verify → Upload**。
2. 開啟Serial Monitor並按RST，核對boot行確實是`debounce_ms=0`。
3. 做5次慢按與5次快速點按，保存從boot開始的完整log。
4. 記錄最後的raw與stable計數。實際做10次，`accepted_press > 10`表示多接受；
   `< 10`表示有按下未被接受；`accepted_release`也要分開檢查。
5. 依序改成10及100 ms重做。30 ms可直接使用練習二同樣10次操作的資料，不必
   為湊次數重做；若練習二操作方式不同，才重新測試。
6. 每次Upload後先核對boot行的`debounce_ms`，避免其實仍在執行上一個版本。
7. 完成後恢復`DEBOUNCE_MS = 30`，重新Verify及Upload，保存最後版本。

**原始log**是從該輪boot行開始、沒有刪除不理想事件的完整Serial文字。只交最後
三行或手動整理後的表格，無法檢查是否多算、漏算或混入前一輪資料。


#### 3-3：整理結果

每輪開始後計數都從0重新開始。若表格中的最後數字不是由該輪boot後開始，資料就
不能直接比較。

| `DEBOUNCE_MS` | 慢按＋快按 | 最後`raw_edge` | 最後`accepted_press` | 最後`accepted_release` | 多接受／漏失 | `stable_for_ms`觀察 | 原始log檔名 |
|---:|---:|---:|---:|---:|---|---|---|
| 0 ms | 5＋5 |  |  |  |  |  |  |
| 10 ms | 5＋5 |  |  |  |  |  |  |
| 30 ms | 5＋5 |  |  |  |  |  |  |
| 100 ms | 5＋5 |  |  |  |  |  |  |

若四輪全部得到10次press與10次release，這不是失敗。正確寫法是：「在本次按鈕、
接線與40次操作中，四種設定都未觀察到stable重複或漏失。」接著指出：這不能證明
0 ms永遠安全，也不能證明其他按鈕、其他操作速度或長期使用都有相同結果。

若0 ms多接受，需回到原始log確認是否同一次實體動作內出現多次stable交替；若
100 ms漏失快速點按，需確認raw是否曾短暫變成true、卻未維持100 ms到stable接受。
如果raw本身完全沒有變化，則優先檢查按鈕與接線，不應直接怪罪去抖設定。


#### 3-4：用證據選擇設定

「最好」不是固定數字，而是在本次需求下平衡兩種風險：

- 等待太短：可能把彈跳當成多次有效事件。
- 等待太長：反應較慢，短暫按壓可能被忽略。

使用自己的表格完成下列句子：

```text
我選擇 ______ ms，因為在本次 ______ 次操作中，
accepted_press = ______、accepted_release = ______，
並且觀察到／未觀察到 ____________________________________。

這個結論只適用於 ________________________________________。
若更換按鈕、接線、操作速度或程式，我需要重新測試，因為
__________________________________________________________。
```

完成條件：選擇必須引用自己的數字，同時說明等待太短及太長的代價，並列出至少
一項不能由本次實驗推廣的情況。不能只寫「30 ms最好」或「網路上都這樣設定」。


三項練習形成同一條證據鏈：練習一用時間軸理解演算法，練習二用自己的按鈕區分
raw與stable，練習三才改變去抖時間並作選擇。完成後可使用
[本notebook附錄](#五延伸實作)選做其他題目；故障現象與安全排查表也集中在附錄。


## 十一、實驗紀錄與繳交內容

建立 Week 2 實驗紀錄（Lab Notebook），至少包含：

1. Arduino IDE board、port、Espressif `esp32` package及PSRAM設定。
2. `version=2`的Serial log，以及Verify、Upload、Reset各能證明什麼。
3. `PIN_BUTTON`、`PIN_TEST_OUTPUT`與GND的接線表及實際GPIO值。
4. 可辨識麵包板座標、按鈕與線路的接線照片。
5. 斷電通斷結果：同列、跨槽、按鈕未按、按住與再次放開。
6. 練習一的去抖時間軸表格，以及使用raw、stable與連續穩定時間完成的三題說明。
7. 練習二30 ms診斷程式的完整log、10次操作表，以及一組raw與stable對應判讀。
8. 練習三的事前預測、0／10／30／100 ms四輪結果表、每輪完整原始log與最後
   的設定選擇及限制。
9. 一項結果與預測不同的地方；若全部符合預測，改寫一項目前資料仍不能證明的事。

Week 2不繳交散裝電阻或直流電壓數字；這些欄位由Week 3教材另行建立。不得只交
「成功」兩個字，也不能只交沒有接腳標示的照片。同組可共用硬體照片，但時間軸
推理、預測、log判讀與結論須由個人完成。


## 十二、完成檢核與器材復原

完成下列項目後，由教師逐項檢查：

- [ ] 能選擇正確Board與Port，完成Verify、Upload、115200-baud Serial及Reset。
- [ ] `PIN_BUTTON`、`PIN_TEST_OUTPUT`與GND接線可追溯到教師公布的profile。
- [ ] 麵包板與按鈕的斷電通斷結果符合實際結構。
- [ ] 能用自己的文字說明接點彈跳、raw讀值、stable狀態及去抖時間。
- [ ] 練習一的時間軸能正確表達raw每次改變都會重新計時。
- [ ] 練習二能從自己的log區分`raw_changed`與`stable_changed`。
- [ ] 練習三四輪只改`DEBOUNCE_MS`，每輪都有可追溯的boot行、原始log與計數。
- [ ] 最後選擇同時說明等待太短、等待太長的代價及本次結論限制。
- [ ] 能指出Serial中的`test_output`是軟體命令，尚不能代替GPIO5電壓實測。
- [ ] 程式、實驗紀錄與個人證據已保存／提交。
- [ ] Serial Monitor已關閉，USB已拔除，萬用電表已轉到`OFF`。
- [ ] 按鈕與杜邦線已清點，板卡無彎針，桌面已復原。

全部通過後，Week 2進度結束，可依課程規定提前離開。散裝電阻、3V3／5Vin、
GPIO LOW／HIGH與Reset安全電壓由Week 3接續，不列為Week 2未完成。未保存證據、
未斷電或未完成收納者，不列為完成。

---


## 十三、附錄：Week 2支援資料

本附錄放置上課前確認表、已購設備辨識索引、延伸實作、故障排查與回報格式。
核心操作步驟請回到[Week 2主教材](#week-2esp32-s3-開發環境與數位輸入輸出實驗)進行。

### 一、本週必帶與器材確認

#### 個人必帶

- [ ] 已安裝Arduino IDE 2與Espressif `esp32` package的筆電。
- [ ] 筆電充電器。
- [ ] 可開啟最新版GitHub課程教材。
- [ ] 一條已確認可傳輸資料的USB線。

#### 每個工作位的實驗依賴

| 品項 | 最低數量 | 本週用途 | 取得方式 |
|---|---:|---|---|
| 經教師核准的ESP32-S3 N16R8板；目前實物為YD-ESP32-S3 Type-A V1.5、排針向下44腳 | 1 | Upload、profile指定輸入與測試輸出 | 學生自備 |
| 可傳資料的USB線 | 1 | 供電、Upload、Serial | 學生自備 |
| 400孔麵包板 | 1 | 按鈕與安全測試點 | 學生自備 |
| 四腳輕觸按鈕 | 1 | 數位輸入 | 學生自備 |
| 公對公杜邦線 | 至少4條 | 兩個profile GPIO、GND及測試點 | 學生自備 |
| 萬用電表 | 輪流共用 | 斷電通斷 | 課堂提供 |

本週不使用LED、蜂鳴器、舵機、馬達、電池盒、外部電源或感測模組。
上電前請把這些物品移出工作區。

實作前另須確認`docs/hardware_state.md`已有本批板卡的target-test組合與GPIO
profile。未公布時，GPIO4／GPIO5只屬候選值，不得依教材文字自行啟用。

#### 上電前最後確認

- [ ] 開發板金屬屏蔽罩與板身絲印已拍照。
- [ ] USB線已確認可傳輸資料。
- [ ] 麵包板、按鈕與杜邦線無明顯損壞。
- [ ] 萬用電表的表筆、電池與通斷／200Ω功能已確認。
- [ ] 桌上沒有外部電池、馬達或高電流負載。

### 二、本週學生器材辨識

本週只辨認學生必買包中的ESP32-S3、麵包板、四腳按鈕
與杜邦線。教師庫存數量及未列入共同必買的感測器、顯示器、馬達與驅動板，
不作為本週學生教材內容。

商品頁截圖只協助核對當時的購買選項，不能證明收到的PCB版型。2026-08-27第一片
實物照片確認模組為Espressif `ESP32-S3-WROOM-1 N16R8`，但開發板PCB是
`VCC-GND Studio YD-ESP32-S3 Type-A V1.5`，不是Espressif原廠DevKitC-1版型。
接腳與USB用途須以YD板實物絲印及後續target test為準。

![商品頁宣稱的ESP32-S3 DevKitC-1 N16R8購買選項](../../docs/images/hardware/products/shopee-esp32-s3-dev-board-n16r8-product-page.png)

第一片實物已拍攝正、反面，並另製作移除桌面的白底辨識圖：

![YD-ESP32-S3 Type-A V1.5正面白底辨識圖](../../docs/images/hardware/actual/yd-esp32-s3-type-a-v1-5-n16r8-actual-front-white-background.png)

![YD-ESP32-S3 Type-A V1.5背面白底辨識圖](../../docs/images/hardware/actual/yd-esp32-s3-type-a-v1-5-n16r8-actual-back-white-background.png)

白底圖用於課堂辨識外形、雙USB接頭、按鈕及排針方向。後製可能改變原本模糊的
細小文字或元件邊緣，因此不能拿來判定模組容量、固定GPIO、電氣規格或焊接品質；
這些判斷必須回看未後製原圖、實物絲印、官方文件及後續target test。

#### 麵包板與三種杜邦線辨識卡

下列白底卡由原始購買截圖後製，並保留蝦皮品名、所選規格、購買數量及訂單金額。
商品外觀經放大整理，只用於課堂辨識，不取代原始訂單、實物或技術規格。
**公頭**末端是露出的金屬針，**母頭**末端是有插孔的黑色塑膠接頭；先看兩端，
不要只靠線的顏色判斷。

##### 400孔麵包板

![400孔麵包板與蝦皮品名、數量及金額](../../docs/images/hardware/product-cards/shopee-breadboard-400-product-card.png)

##### 公對母杜邦線

左端為露出的公針，右端為有插孔的母頭。

![20cm公對母杜邦線與蝦皮品名、規格、數量及金額](../../docs/images/hardware/product-cards/shopee-jumper-wire-20cm-male-to-female-product-card.png)

##### 母對母杜邦線

兩端都是有插孔的母頭，沒有露出的金屬公針。

![20cm母對母杜邦線與蝦皮品名、規格、數量及金額](../../docs/images/hardware/product-cards/shopee-jumper-wire-20cm-female-to-female-product-card.png)

##### 公對公杜邦線

兩端都是露出的金屬公針。

![20cm公對公杜邦線與蝦皮品名、規格、數量及金額](../../docs/images/hardware/product-cards/shopee-jumper-wire-20cm-male-to-male-product-card.png)

BOARD-T01與400孔麵包板已實測：左排落在`B3～B24`、右排落在`J3～J24`，
右側沒有可用外側孔，因此Week 2不把開發板直接壓入單片麵包板。板身排針到
麵包板使用公對母杜邦線；麵包板內部延伸測點可使用公對公杜邦線。本週不需要
母對母杜邦線。

其餘學生必買品項及購買圖片見
[Week 1正式採購總表](../Week_01_Course_Orientation/week1_support.md#purchase-table)。

#### 使用 Windows 相機拍攝與板卡照片檢核

板卡照片用來保存實物辨識證據，不取代官方pinout或實機測試。開始前先拔除
ESP32的USB，移除所有杜邦線與外接電池，並將板卡放在乾燥、不導電的淺色紙面。

##### 開啟相機

1. 按Windows鍵，輸入`相機`或`Camera`，開啟Windows **Camera** App。
2. 第一次開啟若出現權限要求，允許Camera App使用相機；本任務不需要錄音。
3. 確認目前是相片模式，不是錄影模式。若有前後鏡頭切換，選擇能看到桌面的
   一般彩色相機，不使用IR人臉辨識畫面。
4. 若畫面全黑，先關閉Teams、Zoom或其他可能占用相機的程式，再到
   **Settings → Privacy & security → Camera**確認相機存取權限。

##### 取得可辨識照片

1. 擦拭相機鏡頭，在板卡兩側補充均勻光線，避免金屬屏蔽罩反光。
2. 先讓整片板卡平行面向鏡頭。若近距離文字模糊，將板卡移遠到絲印清楚，拍攝
   高解析度照片後再裁切，不要只把模糊板卡放大。
3. 使用畫面上的相片快門按鈕，依序拍攝：
   - 板卡正面全景：完整看見兩排排針、金屬屏蔽罩、USB端與按鈕。
   - USB端近照：看見兩個接頭、BOOT、RESET／RST及周圍絲印。
   - 板卡背面或側面：看見排針方向、兩側針數及是否有彎針。
4. 每拍一張立即開啟縮圖檢查。看不清金屬罩文字、腳位絲印或針腳時重新拍攝，
   不把「肉眼看得到」當成照片已合格。

若筆電Webcam即使移動距離仍無法對焦，不要持續提交模糊照片。改用手機後鏡頭：

1. 將斷電板卡平放在白紙上，使用一般相片模式並增加環境光。
2. 在手機畫面點一下板卡文字位置，使相機對焦；先用原始倍率拍攝，不用數位放大
   製造模糊畫面。
3. 拍完放大檢查絲印，再以USB檔案傳輸、OneDrive、Windows Phone Link或
   Quick Share傳到筆電。避免經過會自動壓縮圖片的聊天軟體。
4. 保存手機產生的原始照片檔，不以社群軟體截圖取代。

Windows通常把相片存到使用者的`Pictures\Camera Roll`；若Pictures由OneDrive
接管，路徑可能顯示為`OneDrive\圖片\Camera Roll`。找不到時，在Camera App開啟
剛拍的縮圖，再選擇在檔案總管中顯示，或依拍攝時間尋找最新相片。

##### 上傳前自我檢核

- [ ] 正面全景沒有裁掉USB端或排針。
- [ ] 金屬屏蔽罩及板面主要文字可放大閱讀。
- [ ] 能分辨兩側各有多少針，並看出排針朝上或朝下。
- [ ] USB接頭、BOOT與RESET／RST按鈕可辨認。
- [ ] 沒有發熱、燒焦、鏽蝕、斷裂或明顯彎針。
- [ ] 畫面沒有姓名、住址、帳號、密碼、學生證或其他個資。

設備核對至少檢查產品是否為ESP32-S3系列開發板、是否符合採購要求的44腳與
排針方向，以及可見型號、USB接頭、按鈕和損傷狀態。照片不能證明GPIO可用、
Flash／PSRAM容量、USB資料通訊或Upload已通過；這些項目必須在後續官方資料
核對、軟體辨識與target test中分別確認。

### 三、缺料或故障回報格式

不要只寫「不能用」。回報時提供：

1. 姓名、作業系統版本與Arduino IDE版本。
2. 開發板模組絲印、板卡正反面照片。
3. 已經完成到主教材的哪一步。
4. 完整錯誤訊息、Port畫面或Serial Monitor截圖。
5. 接線俯視圖，需清楚看到ESP32腳位絲印。
6. 已嘗試的方法與結果。

編譯失敗、上傳失敗、Serial無輸出與按鈕讀值錯誤是
不同問題，必須先指出失敗發生在哪一階段。

### 四、相關資料

- [Week 1正式採購總表](../Week_01_Course_Orientation/week1_support.md#purchase-table)
- [Week 2課前環境準備](../Week_01_Course_Orientation/week1_support.md#week-2-preclass-setup)
- [程式片段](../../docs/course_materials/starter_code_snippets.md)
- [安全檢核](../../docs/course_materials/rubrics_and_checklists.md)

### 五、延伸實作

完成主教材的必要練習後，可選擇下列題目繼續修改、預測、測試與記錄。

#### 延伸實作1：切換模式

每次按下按鈕，`PIN_TEST_OUTPUT`在HIGH與LOW之間切換；放開按鈕不改變模式。

預期log：

```text
event=mode_changed mode=ON test_output=HIGH
event=mode_changed mode=OFF test_output=LOW
```

提示：建立`bool outputOn`，只在新的按下事件發生時反轉它。

#### 延伸實作2：閒置提醒

一段時間都沒有按鈕事件時，輸出一次idle訊息；再次操作後重新計時。

```text
status=idle idle_ms=10000
```

提示：保存`lastActivityMs`。為避免每次loop都重複印出idle，再增加一個
`idleReported`狀態。

#### 延伸實作3：三段狀態循環

每次按下依序切換：

```text
NORMAL -> WARNING -> ALARM -> NORMAL
```

每個狀態要有不同的`PIN_TEST_OUTPUT`行為或Serial文字，並能在Reset後回到NORMAL。

提示：可以使用整數0、1、2，也可以使用`enum`建立有名稱的有限狀態。

#### 延伸實作4：反應時間遊戲

Reset後等待一段隨機時間，Serial顯示`GO`才可按按鈕。記錄從`GO`到
按下的反應時間；提早按下則顯示`too_early`。

```text
game=ready
game=go
game=result reaction_ms=418
```

提示：需要WAIT、GO、RESULT等狀態，並使用`random()`與`millis()`；不可
使用長時間`delay()`，否則無法偵測提早按下。

#### 延伸實作5：加入事件序號

替每一筆按鈕事件加入從1開始的`seq`，讓閱讀log的人能判斷是否漏掉或
重複一筆事件：

```text
group=03 seq=1 event=button_changed pressed=true test_output=HIGH time_ms=12345
group=03 seq=2 event=button_changed pressed=false test_output=LOW time_ms=13021
```

提示：建立`unsigned long eventSeq = 0;`，只在穩定狀態真正改變時先加1，
再把`eventSeq`放進同一行`Serial.printf()`。按下與放開都算一筆事件。

#### 延伸實作6：找出系統無法偵測的故障

先斷電，拔掉`PIN_BUTTON`訊號線，再重新上電。觀察程式會把它看成什麼狀態，
回答：只有`INPUT_PULLUP`時，程式能否分辨「真的沒按」與「訊號線脫落」？

提出一種未來可改善的硬體或軟體方法。本題不要求立刻加購或重新接線。

#### 延伸實作7：交換操作與隱藏錯誤

由另一位組員操作上傳及測試。接著在斷電狀態下，由教師或其他組製造一個
安全的小錯誤，例如換錯`PIN_BUTTON`訊號線的位置。依序使用接線表、通斷、Serial
及電壓證據找出問題；一次只能改一個變因。

#### 延伸實作紀錄

| 選擇的延伸實作 | 修改內容 | 預期結果 | 實際結果 | 修正 |
|---|---|---|---|---|
|  |  |  |  |  |
|  |  |  |  |  |

### 六、故障排查表

| 現象 | 優先檢查 | 禁止或不建議的動作 |
|---|---|---|
| 沒有Port | 資料線、USB接頭、裝置管理員 | 安裝來源不明的driver |
| Compile失敗 | 第一個錯誤、括號、分號、board package | 重插所有接線 |
| Upload失敗 | Board、Port、線材、是否被其他程式占用 | 直接更換整塊板 |
| Upload完成但無Serial | baud rate、port、RESET、`Serial.begin` | 同時改多個設定 |
| 按鈕永遠未按 | `PIN_BUTTON`、GND、按鈕方向、`INPUT_PULLUP` | 帶電改線 |
| 按鈕永遠按下 | `PIN_BUTTON`是否持續短接GND | 將5V接入測試 |
| `Ω 200`時表筆分開卻顯示最左側`1` | 這是開路／超量程的正常表示，不是1Ω | 因看到`1`就判定電表故障 |
| 表筆短接仍顯示數十至上百Ω | 確認小數點、以金屬尖端穩定接觸、插緊兩個表筆接頭，再分別輕動表筆線 | 只聽蜂鳴便記成0Ω，或打磨、拆開電表 |
| 按鈕放開為`1`、按下先顯示約`5x Ω` | 先用麵包板固定按鈕與延伸測點；穩定後應降到`0.x`或數Ω | 單手強壓三個物件，或直接判定按鈕有50Ω |
| 把`10A`看成`10V` | 核對單位：`A`是電流，`V`是電壓；紅表筆仍在`VΩmA` | 把紅表筆移到未熔絲保護的`10A`孔 |
| 只依藍色或黑色選檔 | 同時讀功能符號、量程數字與單位 | 把顏色當成通用安全分類 |
| 板子發熱或異味 | 立即拔USB，通知教師 | 再次上電測試 |

故障排除時一次只改一個變因，並保存修改前後的log，以判定有效的修正動作。

### 七、軟體環境驗證紀錄

| 驗證日期 | 作業系統 | Arduino IDE | Espressif `esp32` package | 驗證範圍 | 結果與限制 |
|---|---|---|---|---|---|
| 2026-08-27 | Windows 64-bit | 2.3.10 | 3.3.11 | 官方下載、`Help → About Arduino IDE`版本確認，以及Boards Manager已安裝狀態 | IDE安裝、啟動與package已安裝狀態已確認；尚未因此宣稱Compile、Upload、Serial或實機線路通過 |

Arduino IDE 發布後仍可能更新。學生應記錄自己實際使用的版本，不能只抄上表；
教師更換課程版本時，也必須重新進行後續 package、Compile、Upload 與 target test。

2026-08-27連線觀察：清楚背面照片顯示兩個接頭分別標為`COM`與`USB`。先前接到
`USB`接頭後，Windows新增「USB序列裝置（COM7）」；這是ESP32-S3原生USB路徑，
不是本週預定的USB-to-UART路徑。此結果只確認本次USB線能供電且Windows完成原生
USB序列裝置列舉。改接背面標示`COM`的接頭後，Windows新增
`USB-Enhanced-SERIAL CH343 (COM8)`（製造商`wch.cn`、USB VID `1A86`、
PID `55D3`），確認這個接頭經過CH343 USB-to-UART橋接晶片；在這項連接埠觀察完成的
當下，Compile、Upload與Serial內容尚未測試，後續驗證結果記錄如下。
`COM7`是這台電腦當次分配結果，不是學生應固定照抄的Port。
同樣地，`COM8`也不是固定答案；學生必須以拔除前後比較辨認自己的Port。

同日使用Arduino-ESP32 `3.3.11`、`ESP32S3 Dev Module`、16 MB Flash、OPI PSRAM、
USB-to-UART及16 MB partition設定編譯
[`week02_board_check.ino`](../../examples/week02_board_check/week02_board_check.ino)成功。Arduino IDE回報程式使用
274505 bytes（8%）、全域變數使用21608 bytes（6%）；容量數字是本次版本與程式的
驗證紀錄，不是學生必須得到的固定數字。此階段只證明compile通過，尚不能宣稱
Upload、Serial或實體板卡功能已通過。

同一設定隨後經CH343 `COM8`以115200 baud完成首次Upload：實際寫入274656 bytes，
寫入資料的hash驗證通過，並出現`Hard resetting via RTS pin...`。整個過程不需手動
按BOOT或RST，確認這塊板的USB-to-UART下載與自動重設路徑可用。此結果仍不等於
Serial輸出、Flash／PSRAM容量或GPIO功能已完成驗證。

第一次runtime輸出確認`chip_model=ESP32-S3`、revision 2、240 MHz、
`flash_bytes=16777216`，且`uptime_ms`每秒增加，證明UART Serial、RESET後重新執行與
16 MB Flash讀值正常。相同輸出中的`psram_bytes=0`不符合N16R8應有的8388608 bytes，
因此PSRAM尚未通過。處理順序是先關閉Serial Monitor並核對
**Tools → PSRAM → OPI PSRAM**，再重新Verify、Upload與讀值；在完成此步驟前不得把
PSRAM記為硬體故障，也不得宣稱N16R8設定已完整驗證。

實際核對發現第一次編譯時PSRAM誤選為`Disabled`。改為`OPI PSRAM`、重新Verify與
Upload後，runtime回報`psram_bytes=8388608`，確認8 MB PSRAM可初始化；問題是選單
設定錯誤，不是硬體故障。這個案例也說明設定畫面、Compile成功與Upload成功都不能
取代runtime讀值。BOARD-T01的完整基本驗證證據見
[2026-08-27 BOARD-T01 basic validation](../../docs/lab_notes/2026-08-27-board-t01-basic-validation.md)。

正面朝上、天線在左、兩個USB接頭在右時，`COM`是右上方、靠近`RX`／`TX`／
`PWR`三個指示燈的接頭；`USB`是右下方、靠近`RGB`區域的接頭。板卡若已旋轉，
仍應翻到背面讀`COM`／`USB`絲印，不靠上下記憶猜測。

### 八、BOARD-T01基本驗證案例如何判讀

本節使用教師實際完成的BOARD-T01驗證，示範同一塊板子需要分層取得證據。
這些數字受到程式、Arduino-ESP32版本與Partition Scheme影響，學生應理解判讀
方法，不抄成自己的固定結果。

#### 1. Board設定不是實物型號

```text
Arduino IDE Board設定：ESP32S3 Dev Module
實際PCB：YD-ESP32-S3 Type-A V1.5
實際模組：ESP32-S3-WROOM-1 N16R8
實際晶片：ESP32-S3
```

`ESP32S3 Dev Module`是通用編譯profile，只指定工具如何替ESP32-S3建立韌體。
它不會把YD板變成原廠DevKitC-1，也不能單獨證明USB位置、固定GPIO或記憶體容量。

#### 2. Verify摘要證明了什麼

本次Verify顯示：

```text
Sketch uses 274505 bytes (8%) of program storage space.
Maximum is 3145728 bytes.
Global variables use 21608 bytes (6%) of dynamic memory,
leaving 306072 bytes for local variables.
Maximum is 327680 bytes.
```

- 274505 bytes是這個版本韌體的程式空間用量。
- 3145728 bytes約為3 MB，是目前Partition Scheme分給應用程式的上限，
  不是整塊板只有3 MB Flash。
- 21608 bytes是編譯時可計算的全域／靜態資料配置；它不是8 MB PSRAM的用量證明。
- 這段訊息證明編譯工具能產生韌體，尚未證明韌體已寫入或能正確執行。

#### 3. Upload摘要證明了什麼

```text
Writing at 0x000530e0 ... 100.0%
Wrote 274656 bytes (157726 compressed) at 0x00010000.
Verifying written data...
Hash of data verified.
Hard resetting via RTS pin...
```

- `Writing at`與百分比表示工具正在向Flash位址寫入資料。
- `compressed`是傳輸時的壓縮量，不等於程式執行時只占相同空間。
- `Hash of data verified`表示寫入後的資料通過本次雜湊比對，不能證明GPIO或程式
  邏輯正確。
- `Hard resetting via RTS pin...`表示CH343路徑自動觸發Reset，使新韌體開始執行；
  它不是錯誤，也不是清除Flash。
- Verify的274505與Upload的274656不必相同；兩段訊息處理的映像、區段、對齊與
  傳輸表示方式不同。

#### 4. Serial Monitor只顯示程式送出的內容

`week02_board_check.ino`在`setup()`中使用`Serial.printf()`讀取並輸出板卡資料，
在`loop()`中每秒輸出一次`millis()`。Serial Monitor不會自行產生以下欄位：

```text
=== Week 2 board check ===
chip_model=ESP32-S3
chip_revision=2
cpu_mhz=240
flash_bytes=16777216
psram_bytes=8388608
status=running
uptime_ms=1018
uptime_ms=2018
uptime_ms=3018
```

| 輸出 | 來源與可支持的判斷 | 不能支持的判斷 |
|---|---|---|
| `chip_model=ESP32-S3` | ESP函式回報執行中的晶片家族 | PCB一定是原廠DevKitC-1 |
| `chip_revision=2` | ESP32-S3晶片silicon revision | PCB版本是V2；它與`Type-A V1.5`不是同一版本編號 |
| `cpu_mhz=240` | 目前程式讀到240 MHz CPU時脈 | 已完成效能或穩定度壓力測試 |
| `flash_bytes=16777216` | runtime讀到16 MiB Flash，符合N16 | 每個Flash區域都已讀寫測試 |
| `psram_bytes=8388608` | OPI PSRAM成功初始化為8 MiB，符合R8 | 整顆PSRAM已完成完整記憶體測試 |
| `status=running` | 程式已執行到這個自訂輸出位置 | ESP32已完成內建的全面健康診斷 |
| `uptime_ms=...` | 本次開機／Reset後經過的毫秒數 | 真實日期時間或跨斷電累積時間 |

Serial Monitor若在`setup()`結束後才開啟，可能只看到後續`uptime_ms`。按一次RST
可讓程式重新執行，標題與各欄位會再次出現，uptime也會從接近0重新累加。
Monitor保留在畫面上的Reset前舊紀錄不代表兩份程式同時執行。

#### 5. `psram_bytes=0`為何不能立刻判定硬體故障

第一次執行時，Compile與Upload都成功，但runtime回報：

```text
psram_bytes=0
```

檢查Tools後發現`PSRAM`誤選為`Disabled`。改成`OPI PSRAM`、重新Verify、Upload
與Reset後才得到8388608。這個修正建立了以下因果鏈：

```text
設定錯誤
→ PSRAM未初始化
→ runtime讀值為0
→ 修正唯一變因並重新上傳
→ runtime讀值成為8388608
```

因此本次問題有證據支持為設定錯誤，而不是PSRAM硬體故障。這也是為什麼排錯時
一次只改一個變因：如果同時更換板子、USB線、Board與PSRAM設定，就無法知道哪一項
真正造成結果改變。

#### 6. 本次完成與尚未完成的範圍

已確認：ESP32-S3程式可編譯與寫入、CH343 USB-to-UART、自動Reset、UART Serial、
16 MB Flash讀值及8 MB OPI PSRAM初始化。

尚未確認：GPIO4／GPIO5輸入輸出、3V3／5Vin實測、GND通斷及任何感測器或
致動器。2026-08-29另以A830L與400孔麵包板抽測一顆四腳按鈕，已建立兩個固定
接點組並確認未按開路、按住導通、放開恢復開路；其餘9顆與GPIO事件仍待驗證。
同日實物對孔確認BOARD-T01為左排`B3～B24`、右排`J3～J24`，單片麵包板直接
安裝沒有右側接線孔，因此正式流程改為板外公對母線。後續每一層仍須以接線、
Serial、萬用電表與實體反應分別取得證據。
